# Action-JEPA — pré-entraînement prédictif et contrôle dans un latent

Ce notebook est le capstone expérimental de `RL_from_scratch`. Il ne cherche pas à reproduire le
ViT-g et le robot Franka de V-JEPA2-AC. Il construit une **maquette de recherche contrôlée** : un
JEPA temporel apprend d'abord une représentation sans action, puis un modèle action-conditionné
apprend la dynamique dans ce latent, enfin CEM/MPC planifie vers un état-but.

La question centrale n'est pas « peut-on refaire Meta avec un MLP ? », mais :

> À données et calcul comparables, vaut-il mieux apprendre simultanément représentation et dynamique,
> ou pré-entraîner la représentation puis la geler comme dans V-JEPA2-AC ?

Nous comparerons donc un entraînement **joint end-to-end** à un entraînement **stage-wise** en trois
phases. Le résultat doit être lisible comme un cours, mais aussi comme une petite étude reproductible.

## Contrat expérimental

Le notebook doit établir une chaîne de preuves, pas seulement faire baisser une loss :

1. la représentation masquée apprend sans collapse ;
2. l'action apporte une information causale au prédicteur ;
3. la loss de rollout limite le drift en boucle ouverte ;
4. la géométrie du latent reste liée au but physique ;
5. CEM fait mieux après apprentissage qu'avec une représentation non entraînée ;
6. les variantes joint et stage-wise sont comparées sur les mêmes données et les mêmes seeds.

Les outputs seront nettoyés pour publication. Les résultats validés devront ensuite être exportés en
figures persistées, conformément à `REGLES.md`.

## Où placer [JEPA](https://openreview.net/pdf?id=BZ5a1r-kVsf) dans la famille des modèles de monde ?

PETS et MBPO apprennent une dynamique directement dans l'espace d'observation: on prédit $o_{t+1}$ à partir de $(o_t, a_t)$. Dreamer apprend un latent, mais ce latent est contraint par un décodeur qui reconstruit l'observation. MuZero apprend aussi un latent sans décodeur, mais il l'entraîne à être **équivalent pour la décision**: reward, value, policy, et recherche.

JEPA propose une autre réponse: on apprend un espace où le futur est prédictible, sans demander au modèle de reconstruire tous les détails visuels ou sensoriels. Pour le contrôle, on ajoute l'action dans le prédicteur: $(z_t, a_t) \rightarrow z_{t+1}$. C'est exactement le pont qui nous intéresse ici: passer d'une représentation auto-supervisée à un modèle de monde utilisable par un planner.

Le même choix de conception sépare les familles : *où* calcule-t-on la perte, et faut-il un décodeur ? JEPA vit dans le latent, sans reconstruction.

```mermaid
flowchart TD
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    q["Modèle de monde :<br/>où calculer la perte ?"]
    inp["Espace d'observation"]
    lat["Espace latent"]
    pets["PETS, MBPO<br/>prédire $$~o_{t+1}$$"]
    dec["avec décodeur"]
    nodec["sans décodeur"]
    dreamer["Dreamer<br/>reconstruit $$~o_{t+1}$$"]
    muzero["MuZero<br/>latent équivalent-décision<br/>$$~(r,v,\pi)$$"]
    jepa["Action-JEPA<br/>prédire $$~E(o_{t+1})$$"]
    q --> inp
    q --> lat
    inp --> pets
    lat --> dec
    lat --> nodec
    dec --> dreamer
    nodec --> muzero
    nodec --> jepa
    classDef hl fill:none,stroke:#2563eb,stroke-width:2px
    class jepa hl
```



## V-JEPA2-AC comme inspiration scientifique

[V-JEPA2](https://arxiv.org/abs/2506.09985) suit une procédure stage-wise : pré-entraînement vidéo
sans action, gel de l'encodeur, post-entraînement d'un prédicteur action-conditionné, puis planning
vers une image-but. Notre adaptation remplace les patchs vidéo par de courts **state tokens** de
Pendulum et le transformer par des MLP lisibles.

Nous gardons aussi la variante end-to-end actuelle. Elle n'est pas présentée comme V-JEPA2-AC : elle
sert de comparaison expérimentale. Sur un petit état vectoriel, laisser le latent s'adapter au contrôle
peut être bénéfique ; à l'inverse, le gel peut stabiliser la dynamique. Le notebook ne présume pas la
réponse : il la mesure.

## Trois phases, deux régimes d'entraînement

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    data["Trajectoires sans labels de tâche"]
    pre["A. JEPA masqué sans action<br/>student partiel, teacher complet"]
    freeze["Geler l'encodeur<br/>$$~E_\star$$"]
    ac["B. Post-training action-conditionné<br/>$$~(z_t,a_t)\rightarrow z_{t+1}$$"]
    plan["C. CEM / MPC<br/>planning vers un latent-but"]
    joint["Comparaison : entraînement joint<br/>représentation + dynamique"]
    data --> pre --> freeze --> ac --> plan
    data --> joint --> plan
    classDef phase fill:none,stroke:#2563eb,stroke-width:2px
    classDef comparison fill:none,stroke:#d97706,stroke-width:2px
    class pre,freeze,ac,plan phase
    class joint comparison
```

Le régime stage-wise sépare clairement « apprendre à représenter » et « apprendre l'effet des
actions ». Le régime joint optimise ces deux problèmes simultanément. Les deux utilisent ensuite le
même planner et le même objectif latent.

## Imports et réglages reproductibles

In [ ]:
import copy
import math
import random
from dataclasses import dataclass, replace
from pathlib import Path

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import trange
from IPython.display import Video, display


plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
print("torch", torch.__version__)

FIGURE_DIR = (
    Path("figures/action_jepa")
    if Path.cwd().name == "notebooks"
    else Path("notebooks/figures/action_jepa")
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def smooth(values, window=20):
    values = np.asarray(values, dtype=np.float32)
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window) / window, mode="valid")


# Environnement — Pendulum-v1

Pendulum donne un problème de contrôle continu compact mais non trivial. L'observation est
$o_t=[\cos\theta_t,\sin\theta_t,\dot\theta_t]$, l'action est un couple dans $[-2,2]$, et la
récompense pénalise angle, vitesse et effort :

$$
r_{t+1}=-\left(\theta_t^2+0.1\dot\theta_t^2+0.001a_t^2\right).
$$

L'épisode est **tronqué** à 200 pas ; il ne possède pas de terminaison physique. Cette distinction
sera importante : la troncature ferme une séquence dans le buffer, mais ne devient pas une cible
`terminal=1`. Le but perceptuel est explicite : $o_{goal}=[1,0,0]$, pendule vertical et immobile.

In [ ]:
INTRO_SEED = 0  # seed locale de cette seule illustration
env = gym.make("Pendulum-v1")
env.action_space.seed(INTRO_SEED)
obs, _ = env.reset(seed=INTRO_SEED)
obs_dim = int(env.observation_space.shape[0])
action_dim = int(env.action_space.shape[0])

print("observation :", np.round(obs, 3), "= [cos(theta), sin(theta), theta_dot]")
print("action      :", env.action_space)
print("horizon     :", env.spec.max_episode_steps, "pas (troncature, pas terminal physique)")

# Un court épisode aléatoire rend immédiatement visibles les données du modèle.
angles, velocities, actions_demo, rewards_demo = [], [], [], []
for _ in range(80):
    action = env.action_space.sample()
    next_obs, reward, terminated, truncated, _ = env.step(action)
    angles.append(math.atan2(next_obs[1], next_obs[0]))
    velocities.append(next_obs[2])
    actions_demo.append(action[0])
    rewards_demo.append(reward)
    if terminated or truncated:
        break

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].plot(angles); axes[0].set_title("Angle aléatoire"); axes[0].set_ylabel("radians")
axes[1].plot(velocities); axes[1].set_title("Vitesse angulaire")
axes[2].plot(actions_demo); axes[2].set_title("Couple appliqué"); axes[2].set_ylim(-2.1, 2.1)
for axis in axes: axis.set_xlabel("pas")
fig.tight_layout()
plt.show()


![Rollout aléatoire Pendulum](figures/action_jepa/pendulum_random_rollout.png)

**Lecture.** L'observation ne contient que trois nombres, mais la tâche exige une stratégie temporelle :
le pendule doit souvent prendre de l'élan avant de rejoindre la verticale. Une prédiction à un pas peut
donc être précise sans être suffisante pour planifier. C'est précisément pourquoi nous mesurerons le
drift multi-pas et utiliserons MPC pour nous recaler sur l'observation réelle.

# Partie I — La philosophie JEPA

Le sigle JEPA signifie *Joint-Embedding Predictive Architecture*. Les trois mots portent chacun une décision de conception, alors décortiquons-les.

- **Joint-embedding** : on ne compare jamais deux objets bruts (deux images, deux observations). On les projette d'abord dans un **espace latent commun** via un encodeur, puis on compare leurs *représentations*. La comparaison vit donc dans l'espace des features, pas dans l'espace des pixels.
- **Predictive** : le modèle ne se contente pas de rapprocher deux vues déjà observées (ça, c'est l'apprentissage par invariance pur). Il doit **prédire** la représentation d'une partie *absente* — masquée, future, ou inaccessible — à partir d'un **contexte** et d'une variable qui indique *quoi* prédire.
- **Architecture** : ce n'est pas une perte isolée mais un assemblage — encodeur de contexte, encodeur cible, et un **prédicteur** entre les deux — dont l'**asymétrie** est ce qui empêche tout l'édifice de s'effondrer.

L'idée centrale se comprend par contraste avec un modèle génératif. Pour prédire la prochaine image d'une vidéo de feuilles dans le vent, un modèle génératif doit **repeindre chaque feuille**, y compris les détails fondamentalement imprévisibles (le bruissement exact de chaque brin). Il dépense énormément de capacité sur ce qui ne se prédit pas. JEPA fait le pari inverse : prédire seulement une **représentation abstraite** du futur (« le feuillage se déplace vers la droite »), et **renoncer** aux détails non prévisibles en ne les représentant tout simplement pas. C'est une façon d'être « incertain » sans payer le prix de la génération.

Cette partie pose la philosophie en trois temps : (1) situer JEPA parmi les familles génératif / contrastif / prédictif, (2) écrire l'équation d'un JEPA **générique**, puis (3) la spécialiser en JEPA **action-conditionné** (JEPA-AC) — la forme dont on a besoin pour le contrôle.

## Génératif, contrastif, prédictif: trois familles à distinguer

Trois grandes familles d'apprentissage auto-supervisé se distinguent par **où** elles calculent leur perte et **comment** elles évitent la solution triviale.

**Génératif / reconstruction.** Le réseau apprend à produire ou reconstruire les données. *Exemple : un auto-encodeur masqué (MAE) cache 75 % des patchs d'une image et reconstruit les pixels manquants.* L'anti-collapse est gratuit (pour reconstruire, il faut bien encoder l'information), mais beaucoup de capacité part dans des détails inutiles à la décision : texture, bruit de capteur, arrière-plan.

**Contrastif (invariance + négatifs).** Le réseau rapproche deux vues *positives* d'un même objet et **repousse** les autres exemples du batch (les *négatifs*). *Exemple : SimCLR rapproche deux recadrages de la même photo et les éloigne de toutes les autres photos du batch.* Très efficace, mais sensible à la fabrique des négatifs : combien en faut-il, et comment éviter les faux négatifs (deux photos différentes du même chat) ?

**Joint-embedding prédictif (JEPA).** Le réseau **prédit la représentation** d'une partie absente via un prédicteur, sans reconstruire les pixels et sans négatifs. *Exemple : [I-JEPA](https://arxiv.org/abs/2301.08243) prédit les features de blocs d'image masqués à partir d'un bloc-contexte visible.* L'anti-collapse vient de l'asymétrie online/cible (EMA + stop-gradient) et, en option, d'une régularisation (VICReg, Partie II).

| Famille | Mécanisme | Perte calculée dans | Anti-collapse | Exemples |
|---|---|---|---|---|
| **Génératif / reconstruction** | reconstruire l'entrée (parfois masquée) | **espace d'entrée** | implicite (il faut tout reproduire) | [AE](https://people.engr.tamu.edu/rgutier/web_courses/cpsc636_s10/kramer1991nonlinearPCA.pdf), [VAE](https://arxiv.org/abs/1312.6114), [MAE](https://arxiv.org/abs/2111.06377) |
| **Contrastif (invariance + négatifs)** | rapprocher deux vues d'un même objet, **éloigner** les autres | espace de représentation | **négatifs** (InfoNCE) | [SimCLR](https://arxiv.org/abs/2002.05709), [MoCo](https://arxiv.org/abs/1911.05722) |
| **Joint-embedding prédictif (JEPA)** | **prédire** la représentation d'une vue/partie via un **prédicteur** | espace de représentation | EMA + prédicteur, ou régularisation ([VICReg](https://arxiv.org/abs/2105.04906)) | [I-JEPA](https://arxiv.org/abs/2301.08243), [V-JEPA](https://arxiv.org/abs/2404.08471), [BYOL](https://arxiv.org/abs/2006.07733) (cas limite) |

**Pour le sentir.** Demandez aux trois familles « prédis la case cachée d'un échiquier en désordre ». Le génératif **dessine** la case et se trompe sur la pièce exacte. Le contrastif dit « cette case ressemble plus à celle-ci qu'à celle-là ». JEPA répond « la *représentation* de cette case est à peu près celle-ci » — sans s'engager sur les pixels. C'est précisément ce dont un planner a besoin : une géométrie du futur, pas un dessin du futur.

## L'équation d'un JEPA (forme générale)

Posons d'abord la forme **générique**, sans action, telle qu'on la trouve dans [I-JEPA](https://arxiv.org/abs/2301.08243) ou [V-JEPA](https://arxiv.org/abs/2404.08471). On dispose d'un **contexte** $x$ et d'une **cible** $y$ (par exemple : $x$ = patchs visibles d'une image, $y$ = patchs masqués). Deux encodeurs entrent en jeu :

$$
s_x = E_\theta(x), \qquad s_y = \operatorname{sg}\big(E_\xi(y)\big)
$$

$E_\theta$ est l'**encodeur online**, entraîné par gradient ; $E_\xi$ est l'**encodeur cible**, une copie lente mise à jour par moyenne mobile (EMA) ; et $\operatorname{sg}$ est le *stop-gradient* — la cible ne reçoit aucun gradient de la perte.

Le **prédicteur** $P_\phi$ reconstruit la représentation de la cible à partir du contexte et d'une **variable de conditionnement** $c$ qui précise *quelle* partie prédire (dans I-JEPA, $c$ encode les positions des blocs masqués) :

$$
\hat s_y = P_\phi(s_x,\, c), \qquad
\mathcal L_{\text{JEPA}} = \big\lVert \hat s_y - s_y \big\rVert_2^2
$$

La même équation se décline ainsi :

- **I-JEPA** (image) : $x$ = bloc-contexte, $y$ = blocs masqués, $c$ = leurs positions.
- **V-JEPA** (vidéo) : idem en spatio-temporel, $y$ = *tubelets* masqués dans le temps.
- **[BYOL](https://arxiv.org/abs/2006.07733)** : cas limite où $x$ et $y$ sont deux augmentations de la *même* image et où $c$ est vide ; il n'y a plus de « quoi prédire », seulement « prédis l'autre vue ». BYOL est donc **architecturalement proche** d'un JEPA (même asymétrie online/cible, même stop-gradient), mais sans partie absente à prédire ni conditionnement : c'est un **cas limite** de la famille, pas un JEPA au sens strict.

La signature de JEPA, commune aux trois : **la perte se calcule entre représentations**, jamais entre $\hat y$ et $y$ bruts. On ne reconstruit rien.

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    x["Contexte visible<br/>$$~x$$"]
    student["Encodeur online / student<br/>$$~E_\theta$$"]
    sx["Représentation du contexte<br/>$$~s_x$$"]

    c["Conditionnement<br/>$$~c$$"]
    predictor["Prédicteur<br/>$$~P_\phi$$"]
    predicted["Représentation prédite<br/>$$~\hat{s}_y$$"]

    y["Cible observée<br/>$$~y$$"]
    teacher["Encodeur cible / teacher<br/>$$~E_\xi$$"]
    stop["Stop-gradient<br/>aucun gradient"]
    target["Représentation cible<br/>$$~s_y$$"]

    loss["Perte dans l’espace latent<br/>$$~\mathcal{L}_{\mathrm{JEPA}}=\lVert\hat{s}_y-s_y\rVert_2^2$$"]

    x --> student --> sx --> predictor --> predicted --> loss
    c --> predictor

    y --> teacher --> stop --> target --> loss


    student -.->|"mise à jour EMA"| teacher

    classDef online fill:none,stroke:#2563eb,stroke-width:2px
    classDef targetBranch fill:none,stroke:#d97706,stroke-width:2px
    classDef stopped fill:none,stroke:#dc2626,stroke-width:2px
    classDef objective fill:none,stroke:#16a34a,stroke-width:2px

    class student,sx,predictor,predicted online
    class teacher,target targetBranch
    class stop stopped
    class loss objective
```

Le student encode le contexte, puis le prédicteur utilise le conditionnement pour anticiper la représentation cible. La loss rapproche cette prédiction du latent produit par le teacher. Le gradient met à jour uniquement le student et le prédicteur ; le teacher reste hors du chemin de rétropropagation et évolue lentement par EMA. On apprend donc à prédire une représentation stable, jamais à reconstruire directement la cible brute.

## De JEPA à JEPA action-conditionné (JEPA-AC)

Pour le contrôle, la variable de conditionnement $c$ devient l'**action**. Le contexte est l'observation courante $o_t$, la cible est l'observation suivante $o_{t+1}$, et le prédicteur répond à « *que devient le latent si je joue $a_t$ ?* ». C'est exactement le passage de I-JEPA/V-JEPA à V-JEPA2-AC.

Pour une transition $(o_t, a_t, o_{t+1})$ :

$$
z_t = E_\theta(o_t), \qquad z^+_{t+1} = \operatorname{sg}\big(E_\xi(o_{t+1})\big), \qquad \hat z_{t+1} = P_\phi(z_t, a_t)
$$

$$
\mathcal L_{\text{JEPA-AC}} = \big\lVert \hat z_{t+1} - z^+_{t+1} \big\rVert_2^2
$$

C'est *littéralement* l'équation générale précédente avec $x = o_t$, $y = o_{t+1}$ et $c = a_t$. Le stop-gradient reste indispensable : sans lui, rien n'empêche les deux encodeurs de s'accorder sur une représentation constante $E(\cdot) = \text{cste}$, qui annule la perte sans rien apprendre — comme deux élèves qui conviendraient d'écrire « 42 » à chaque question : toujours d'accord, mais aucun savoir.

*Exemple concret (Pendulum).* Ici $o_t = (\cos\theta, \sin\theta, \dot\theta)$ et $a_t$ est le couple moteur. $E_\theta$ encode l'état, et $P_\phi$ prédit le latent de l'état suivant **sachant le couple appliqué**. Une fois ce modèle appris, planifier revient à chercher la séquence de couples dont les latents imaginés se rapprochent du latent-but (pendule vertical) — sans jamais redessiner d'image. C'est la maquette de l'idée V-JEPA2-AC de planifier vers une image-but dans l'espace latent.


```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    obs["Observation courante<br/>$$~o_t$$"]
    student["Encodeur online / student<br/>$$~E_\theta$$"]
    latent["Latent courant<br/>$$~z_t$$"]

    action["Action exécutée<br/>$$~a_t$$"]
    predictor["Prédicteur action-conditionné<br/>$$~P_\phi$$"]
    predicted["Prochain latent prédit<br/>$$~\hat{z}_{t+1}$$"]

    nextObs["Observation suivante<br/>$$~o_{t+1}$$"]
    teacher["Encodeur cible / teacher<br/>$$~E_\xi$$"]
    stop["Stop-gradient<br/>aucun gradient"]
    target["Prochain latent cible<br/>$$~z^+_{t+1}$$"]

    loss["Perte de dynamique latente<br/>$$~\mathcal{L}_{\mathrm{JEPA-AC}}=\lVert\hat{z}_{t+1}-z^+_{t+1}\rVert_2^2$$"]

    obs --> student --> latent --> predictor --> predicted --> loss
    action --> predictor

    nextObs --> teacher --> stop --> target --> loss

    student -.->|"mise à jour EMA"| teacher

    classDef online fill:none,stroke:#2563eb,stroke-width:2px
    classDef actionNode fill:none,stroke:#7c3aed,stroke-width:2px
    classDef targetBranch fill:none,stroke:#d97706,stroke-width:2px
    classDef stopped fill:none,stroke:#dc2626,stroke-width:2px
    classDef objective fill:none,stroke:#16a34a,stroke-width:2px

    class student,latent,predictor,predicted online
    class action actionNode
    class teacher,target targetBranch
    class stop stopped
    class loss objective
```
La branche online encode l’état présent, puis le prédicteur combine ce latent avec l’action pour imaginer le latent suivant. En parallèle, le teacher encode l’observation réellement obtenue après cette action. La loss compare donc la conséquence imaginée de l’action à la conséquence réellement observée, mais dans l’espace latent. Le teacher ne reçoit aucun gradient et suit lentement le student par EMA. Une fois cette dynamique apprise, on peut tester plusieurs actions sans les exécuter réellement, puis choisir celle dont le futur latent paraît le plus favorable.

## Pourquoi l'action change la nature du problème

Dans I-JEPA ou V-JEPA, le contexte prédit une partie masquée ou future, mais le futur y est **passif** : il ne dépend que de ce qui est déjà là. En contrôle, le futur est **agent-dépendant** : depuis le même état, deux actions différentes mènent à deux futurs différents.

*Exemple.* Pendule à l'horizontale, immobile. Couple $a_t = +2$ : il bascule d'un côté, ce qui donne un certain $z^+_{t+1}$. Couple $a_t = -2$ : il part de l'autre côté, donc un $z^+_{t+1}$ très différent. Un prédicteur **sans** action ne pourrait, au mieux, prédire que la *moyenne* de ces futurs — une bouillie inutilisable pour planifier. En injectant $a_t$, le prédicteur sépare les conséquences et devient un véritable **modèle de dynamique latent**.

C'est cette propriété qui rend le modèle *planifiable* : on peut dérouler mentalement plusieurs séquences d'actions dans le latent, les comparer, puis n'exécuter que la meilleure première action. On passe de « que va-t-il arriver ? » à « que va-t-il arriver **si je fais ça** ? » — la question de tout planner.

Un test pratique en découle directement (Partie V, *sensibilité à l'action*) : si mélanger les actions dans un batch ne change *pas* l'erreur de prédiction, c'est que le prédicteur **ignore** l'action — il a appris l'inertie de l'observation, pas une dynamique contrôlable.

## Un petit problème jouet pour sentir la prédiction latente

Avant Pendulum, on crée des paires synthétiques en deux dimensions. La cible est une version légèrement tournée et bruitée de l'entrée. Ce jouet n'est pas un environnement RL, mais il permet d'observer rapidement le comportement des mécanismes anti-collapse sans attendre une longue simulation.

On définit aussi un petit encodeur, une fonction de mesure `latent_std`, et une mise à jour EMA. Ces trois briques reviennent ensuite dans le vrai modèle: encodeur en ligne, encodeur cible et diagnostic de dispersion du latent.

In [ ]:
def toy_pairs(n=512, theta=0.35):
    x = torch.randn(n, 2)
    rot = torch.tensor([[math.cos(theta), -math.sin(theta)], [math.sin(theta), math.cos(theta)]], dtype=torch.float32)
    return x, x @ rot.T + 0.05 * torch.randn_like(x)

class TinyEncoder(nn.Module):
    def __init__(self, input_dim=2, hidden=32, latent=8):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.SiLU(), nn.Linear(hidden, latent))
    def forward(self, x):
        return self.net(x)

def latent_std_value(z):
    return float(z.std(dim=0).mean().detach().cpu())

@torch.no_grad()
def ema_update_module(target, online, tau):
    for pt, po in zip(target.parameters(), online.parameters()):
        pt.mul_(tau).add_(po.detach(), alpha=1.0 - tau)

**Mini-test.** Vérification visuelle du `toy exemple`. Une simple vérification de shape ne montre pas la transformation que le modèle devra apprendre. On génère donc plusieurs centaines de paires et on vérifie trois propriétés :
chaque cible correspond bien à son entrée ;
la rotation observée est proche de l’angle imposé $\theta=0.35$ rad ;
la cible contient un petit bruit résiduel, donc elle n’est pas une copie déterministe parfaite.
La figure de gauche relie quelques entrées à leurs cibles. Celle de droite mesure l’angle de rotation de chaque paire.

Dans les notebooks pédagogiques, ces mini-tests ne remplacent pas la suite de tests du package. Ils jouent un autre rôle: empêcher le lecteur de continuer avec une brique mentale ou logicielle cassée. Quand on introduit un mécanisme abstrait comme JEPA, ces points d'ancrage sont très utiles.

In [ ]:
# Vérification réelle du jouet : rotation connue + petit bruit
torch.manual_seed(0)

theta_expected = 0.35
x_demo, y_demo = toy_pairs(n=512, theta=theta_expected)

# Transformation déterministe attendue, avant ajout du bruit gaussien.
rotation = torch.tensor(
    [
        [math.cos(theta_expected), -math.sin(theta_expected)],
        [math.sin(theta_expected),  math.cos(theta_expected)],
    ],
    dtype=torch.float32,
)
y_without_noise = x_demo @ rotation.T
residual_noise = y_demo - y_without_noise

# Angle signé entre chaque entrée et sa cible.
angle_x = torch.atan2(x_demo[:, 1], x_demo[:, 0])
angle_y = torch.atan2(y_demo[:, 1], y_demo[:, 0])
angle_delta = torch.atan2(
    torch.sin(angle_y - angle_x),
    torch.cos(angle_y - angle_x),
)

# Les points proches de l'origine ont un angle très sensible au bruit.
reliable = x_demo.norm(dim=1) > 0.5
median_rotation = float(angle_delta[reliable].median())
noise_rmse = float(residual_noise.pow(2).mean().sqrt())
mean_pair_distance = float((y_demo - x_demo).norm(dim=1).mean())

assert x_demo.shape == y_demo.shape == (512, 2)
assert not torch.allclose(x_demo, y_demo)
assert abs(median_rotation - theta_expected) < 0.04
assert 0.03 < noise_rmse < 0.07

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# Gauche : quelques correspondances x -> y.
n_arrows = 45
axes[0].scatter(
    x_demo[:n_arrows, 0],
    x_demo[:n_arrows, 1],
    s=35,
    color="#2563eb",
    label="entrée $x$",
    zorder=3,
)
axes[0].scatter(
    y_demo[:n_arrows, 0],
    y_demo[:n_arrows, 1],
    s=35,
    color="#f97316",
    label="cible $y$",
    zorder=3,
)

for x_point, y_point in zip(x_demo[:n_arrows], y_demo[:n_arrows]):
    axes[0].annotate(
        "",
        xy=y_point.numpy(),
        xytext=x_point.numpy(),
        arrowprops=dict(
            arrowstyle="->",
            color="#64748b",
            alpha=0.45,
            lw=0.9,
        ),
    )

axes[0].axhline(0, color="black", lw=0.7, alpha=0.4)
axes[0].axvline(0, color="black", lw=0.7, alpha=0.4)
axes[0].set_aspect("equal")
axes[0].set_title("Chaque cible est l’entrée tournée et bruitée")
axes[0].set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")
axes[0].legend()

# Droite : distribution de l'angle effectivement observé.
axes[1].hist(
    angle_delta[reliable].numpy(),
    bins=35,
    color="#93c5fd",
    edgecolor="white",
    alpha=0.9,
)
axes[1].axvline(
    theta_expected,
    color="#dc2626",
    ls="--",
    lw=2,
    label=f"rotation attendue = {theta_expected:.2f} rad",
)
axes[1].axvline(
    median_rotation,
    color="#1d4ed8",
    lw=2,
    label=f"médiane observée = {median_rotation:.2f} rad",
)
axes[1].set_title("La rotation reste identifiable malgré le bruit")
axes[1].set_xlabel("angle entre $x$ et $y$ (radians)")
axes[1].set_ylabel("nombre de paires")
axes[1].legend()

fig.tight_layout()
plt.show()

print(f"Rotation imposée          : {theta_expected:.3f} rad")
print(f"Rotation médiane observée : {median_rotation:.3f} rad")
print(f"RMSE du bruit ajouté      : {noise_rmse:.3f}")
print(f"Distance moyenne ||y-x||  : {mean_pair_distance:.3f}")

**Lecture**. À gauche, chaque flèche relie une entrée bleue à sa cible orange : les déplacements suivent globalement la même rotation, tout en présentant de petites variations dues au bruit. À droite, les angles mesurés se concentrent autour de $0.35$ rad, ce qui confirme que la transformation possède une structure prédictible.
Le problème n’est donc ni trivial ni aléatoire : la rotation constitue le signal que le modèle doit apprendre, tandis que le bruit représente la partie imprévisible. Un bon JEPA doit conserver la structure commune dans son latent sans chercher à reproduire exactement chaque perturbation aléatoire.

# Partie II — Le collapse : le piège central

Le **collapse** est une solution mathématiquement correcte mais totalement inutile : l’encodeur renvoie presque le même vecteur pour toutes les observations,

$$
E(o_1)\approx E(o_2)\approx\cdots\approx c.
$$

Le prédicteur n’a alors plus besoin de comprendre la dynamique. Il lui suffit de produire cette même constante :

$$
P_\phi(c,a)\approx c.
$$

La perte de prédiction devient presque nulle, quelles que soient l’observation et l’action. Pourtant, le latent ne permet plus de distinguer un pendule vertical d’un pendule tombé. Pour un planner, tous les futurs semblent identiques : la représentation est prédictible, mais elle ne contient plus aucune information utile.

Il existe en réalité deux formes de collapse :

- **collapse complet** : toutes les observations deviennent presque le même vecteur ;
- **collapse dimensionnel** : certaines dimensions restent variables, mais beaucoup deviennent constantes ou se recopient mutuellement.

Le second cas est plus discret. Le latent peut conserver une variance globale non nulle tout en n’utilisant réellement qu’un ou deux axes. C’est pourquoi surveiller uniquement la loss JEPA ne suffit pas : il faut aussi mesurer la dispersion et la redondance de la représentation.

Notre Action-JEPA combine donc deux niveaux de protection :

1. une asymétrie architecturale : prédicteur uniquement côté online, *stop-gradient* et cible EMA ;
2. une protection explicite : VICReg-lite impose de la variance et limite la redondance entre dimensions.


## La voie contrastive : repousser des négatifs

La solution historique au collapse consiste à utiliser des **exemples négatifs**. Pour une représentation $z_i$, on définit :

- un positif $z_i^+$, obtenu à partir du même objet ou de la même scène ;
- des négatifs $z_j$, provenant des autres exemples du batch.

L’objectif rapproche la paire positive tout en éloignant les négatifs. Une forme simplifiée d’InfoNCE est :

$$
\mathcal L_{\text{InfoNCE}}
=
-\log
\frac{
\exp\!\left(\operatorname{sim}(z_i,z_i^+)/\tau\right)
}{
\sum_j \exp\!\left(\operatorname{sim}(z_i,z_j)/\tau\right)
}.
$$

Le numérateur mesure la similarité avec la bonne cible. Le dénominateur compare cette cible à toutes les représentations candidates du batch. La loss peut donc se lire comme une classification : *parmi toutes ces représentations, laquelle correspond réellement à $z_i$ ?*

Le paramètre $\tau$ est une **température** :

- une petite valeur accentue fortement les différences de similarité ;
- une grande valeur produit une distribution plus douce.

Pourquoi cela empêche-t-il le collapse ? Si toutes les observations donnent le même latent, toutes les similarités sont identiques. Le modèle ne peut plus reconnaître le positif parmi les $B$ candidats : sa probabilité vaut environ $1/B$, et sa loss reste proche de $\log B$. Une représentation constante n’est donc pas une solution satisfaisante.

Cette approche fonctionne très bien dans SimCLR ou MoCo, mais elle déplace le problème vers la construction des négatifs :

- combien de négatifs faut-il ?
- doivent-ils venir du batch ou d’une mémoire externe ?
- que se passe-t-il si un « négatif » représente en réalité la même classe ou le même état sémantique ?

Ce dernier cas est un **faux négatif** : le modèle repousse deux observations qui devraient rester proches. Les méthodes joint-embedding comme BYOL, SimSiam et JEPA cherchent justement à apprendre sans dépendre de cette opposition explicite.


## La voie BYOL/SimSiam : asymétrie, stop-gradient et prédicteur

BYOL a montré qu’un modèle pouvait apprendre sans négatifs explicites grâce à une architecture **asymétrique**. Deux encodeurs produisent des représentations :

- l’encodeur **online** $E_\theta$, entraîné par gradient ;
- l’encodeur **cible** $E_\xi$, utilisé comme teacher lent.

Un prédicteur $P_\phi$ existe uniquement sur la branche online. Le modèle optimise donc une relation asymétrique :

$$
P_\phi(E_\theta(x))
\approx
\operatorname{sg}\!\left(E_\xi(y)\right).
$$

Le symbole $\operatorname{sg}$ signifie *stop-gradient*. La représentation cible intervient dans la loss, mais aucun gradient ne traverse cette branche. Le teacher ne cherche donc pas à poursuivre instantanément la prédiction du student.

Ses paramètres suivent lentement l’encodeur online par moyenne mobile exponentielle :

$$
\xi \leftarrow \tau\xi+(1-\tau)\theta.
$$

Avec $\tau=0.99$, la nouvelle cible conserve 99 % de son ancienne valeur et incorpore seulement 1 % des paramètres online. Elle joue ainsi le rôle d’un repère qui évolue lentement :

- si $\tau$ est trop faible, la cible suit presque immédiatement le student et devient instable ;
- si $\tau$ est très proche de 1, elle est très stable, mais s’adapte plus lentement.

Dans Action-JEPA, cette mécanique devient :

$$
P_\phi\big(E_\theta(o_t),a_t\big)
\approx
\operatorname{sg}\!\left(E_\xi(o_{t+1})\right).
$$

Le prédicteur doit donc expliquer le changement de représentation à partir de l’action, tandis que la cible reste momentanément fixe.

Il faut néanmoins rester précis : EMA, stop-gradient et prédicteur rendent le collapse beaucoup moins attractif, mais ils ne constituent pas une garantie universelle dans tous les petits réseaux et tous les régimes d’entraînement. C’est pourquoi notre version ajoute VICReg-lite : au lieu d’espérer seulement que le latent reste informatif, on le vérifie et on l’encourage explicitement.


## VICReg-lite : préserver explicitement l’information

VICReg décompose la santé d’une représentation en trois propriétés :

1. **invariance** : deux vues correspondantes doivent produire des représentations proches ;
2. **variance** : chaque dimension doit continuer à varier entre les exemples ;
3. **covariance** : les dimensions ne doivent pas toutes recopier la même information.

Dans notre Action-JEPA, la perte de prédiction joue un rôle **analogue** à l’invariance — elle aligne le latent prédit sur la cible — sans en être *littéralement* le terme VICReg (qui rapproche deux vues du même instant) :

$$
\mathcal L_{\text{rollout}}
=
\frac{1}{K}
\sum_{k=1}^{K}
\left\|
\hat z_{t+k}-z^+_{t+k}
\right\|_2^2.
$$

On ajoute donc principalement les termes de variance et de covariance.

### Préserver la variance

Pour chaque dimension latente $j$, on mesure son écart-type dans le batch :

$$
\sigma_j=\operatorname{std}(z_{:,j}).
$$

On pénalise uniquement les dimensions dont la dispersion descend sous un seuil $\gamma$ :

$$
\mathcal L_{\text{var}}
=
\frac{1}{d}
\sum_{j=1}^{d}
\max(0,\gamma-\sigma_j).
$$

Si $\sigma_j\geq\gamma$, la dimension est suffisamment active et ne reçoit aucune pénalité. Si $\sigma_j\approx0$, elle est presque constante : la pénalité devient forte.

Ce terme ne demande pas que les valeurs soient grandes sans limite. Il impose seulement un **plancher de dispersion** afin que différentes observations restent distinguables.

### Réduire la covariance

Une représentation peut cependant éviter le collapse complet tout en recopiant la même variable sur tous ses axes :

$$
z_2\approx z_1,\qquad z_3\approx z_1,\qquad \ldots
$$

Le latent varie, mais sa dimension effective reste proche de 1. On calcule donc sa matrice de covariance :

$$
\operatorname{Cov}(z)
=
\frac{1}{B-1}
(z-\bar z)^\top(z-\bar z).
$$

La diagonale contient la variance de chaque dimension. Les termes hors diagonale indiquent si deux dimensions évoluent ensemble. VICReg pénalise ces termes :

$$
\mathcal L_{\text{cov}}
=
\frac{1}{d}
\sum_{i\neq j}
\operatorname{Cov}(z)_{ij}^{\,2}.
$$

Cette pénalité encourage les dimensions à porter des informations différentes. Elle ne garantit pas une indépendance statistique complète : elle réduit seulement leur redondance **linéaire**.

Un exemple permet de distinguer les deux pertes. Supposons :

$$
z_1 \text{ varie},\qquad z_2=z_1,\qquad z_3=\text{constante}.
$$

- $\mathcal L_{\text{var}}$ pénalise surtout $z_3$, qui ne varie pas ;
- $\mathcal L_{\text{cov}}$ pénalise la redondance entre $z_1$ et $z_2$.

Les deux termes sont donc complémentaires : la variance empêche les axes de mourir, tandis que la covariance empêche les axes survivants de tous raconter la même chose.




## Implémenter variance et covariance

Dans le code, `variance_loss` calcule l’écart-type de chaque colonne du tenseur latent de forme `[batch, latent_dim]`, puis applique la pénalité sous le seuil. `covariance_loss` centre le batch, construit la matrice de covariance et met sa diagonale à zéro avant de sommer les carrés restants.

La division par $d$ suit la formulation de VICReg : on normalise par le nombre de dimensions latentes, et non par les $d^2$ éléments de la matrice. Une moyenne naïve sur toute la matrice sous-pondérerait le terme de covariance d’un facteur proche de $d$.

Ces pertes ont un double rôle dans le notebook :

- **optimisation** : elles poussent l’encodeur à conserver une représentation exploitable ;
- **diagnostic** : elles permettent de détecter un modèle dont la loss de prédiction baisse pour de mauvaises raisons.

C’est un point central : une faible erreur JEPA signifie seulement que la cible est facile à prédire. Elle ne garantit pas que la cible soit informative. La variance et la covariance servent précisément à vérifier que le modèle n’a pas rendu son propre problème artificiellement facile.

In [ ]:
def variance_loss(z, target_std=1.0, eps=1e-4):
    std = torch.sqrt(z.var(dim=0, unbiased=False) + eps)
    return F.relu(target_std - std).mean()

def covariance_loss(z):
    z = z - z.mean(dim=0, keepdim=True)
    cov = (z.T @ z) / max(1, z.shape[0] - 1)
    off = cov - torch.diag(torch.diag(cov))
    # VICReg : somme des termes hors-diagonale au carré, divisée par d
    # (et non .mean() sur d*d, qui sous-pondère le terme d'un facteur 1/d).
    return off.pow(2).sum() / z.shape[1]

## Illustration : vitesse EMA et stabilité du latent

On entraîne maintenant le problème jouet avec différentes valeurs de $\tau$, le coefficient de la moyenne mobile exponentielle :

$$
\xi_t=\tau\xi_{t-1}+(1-\tau)\theta_t.
$$

Cette formule détermine la vitesse à laquelle le teacher $E_\xi$ suit le student $E_\theta$ :

- avec $\tau=0$, la cible est immédiatement remplacée par l’encodeur online ;
- avec $\tau=0.9$, elle incorpore 10 % des nouveaux paramètres à chaque update ;
- avec $\tau=0.99$, elle n’en incorpore que 1 % et résume grossièrement une centaine d’updates ;
- avec $\tau$ très proche de 1, elle devient un repère très stable, mais peut prendre du retard sur l’encodeur online.

On peut voir l’EMA comme un professeur qui met progressivement à jour son corrigé. S’il le réécrit entièrement après chaque réponse de l’élève, la cible bouge trop vite et ne fournit plus de repère stable. S’il ne le modifie presque jamais, le corrigé devient obsolète. Le bon $\tau$ équilibre donc **stabilité** et **capacité d’adaptation**.

Pour observer l’effet de ce réglage, la figure trace la dispersion moyenne du latent :

$$
\operatorname{latent\_std}(z)
=
\frac{1}{d}
\sum_{j=1}^{d}
\operatorname{std}(z_{:,j}).
$$

Pour chaque dimension $j$, on mesure l’écart-type de ses valeurs dans le batch, puis on moyenne ces écarts-types. Si toutes les observations sont encodées presque au même endroit, chaque $\operatorname{std}(z_{:,j})$ tend vers zéro et le diagnostic chute : c’est le signal d’un collapse complet.

À l’inverse, un `latent_std` non nul montre que le modèle continue à distinguer les observations. Il ne prouve toutefois pas que la représentation soit bonne : plusieurs dimensions peuvent être redondantes, ou varier sans contenir l’information utile à l’action. C’est pourquoi ce diagnostic doit être lu avec la perte de covariance, la sensibilité à l’action et, plus tard, la performance du planner.

La figure ne cherche donc pas à désigner un $\tau$ universellement optimal. Elle montre comment la vitesse de la cible influence la stabilité de l’apprentissage et fournit une première alarme : une loss JEPA faible accompagnée d’un `latent_std` proche de zéro serait une réussite seulement en apparence.

In [ ]:
def train_ema_demo(tau, steps=160, use_vicreg=False):
    torch.manual_seed(0)
    online = TinyEncoder()
    target = copy.deepcopy(online)
    for p in target.parameters():
        p.requires_grad_(False)

    predictor = nn.Sequential(nn.Linear(8, 32), nn.SiLU(), nn.Linear(32, 8))
    opt = torch.optim.Adam(list(online.parameters()) + list(predictor.parameters()), lr=1e-3)
    stds = []

    for _ in range(steps):
        x, y = toy_pairs()
        pred_z = predictor(online(x))
        with torch.no_grad():
            tgt_z = target(y)

        # JEPA/BYOL-style objective: predict a slowly moving target representation.
        loss = F.mse_loss(pred_z, tgt_z)
        if use_vicreg:
            z = online(x)
            loss = loss + variance_loss(z, 1.0) + 0.04 * covariance_loss(z)

        opt.zero_grad()
        loss.backward()
        opt.step()
        ema_update_module(target, online, tau)
        stds.append(latent_std_value(online(x)))
    return np.asarray(stds)

plt.figure(figsize=(7, 3.5))
for tau in [0.0, 0.5, 0.9, 0.99]:
    plt.plot(train_ema_demo(tau), label=f"tau={tau}")
plt.axhline(0.1, color="r", ls=":", label="zone collapse")
plt.xlabel("étape")
plt.ylabel("latent_std")
plt.legend()
plt.title("Vitesse EMA et stabilité du latent")
plt.show()

**Lecture.** La courbe ne doit pas être interprétée comme une loi universelle sur $\tau$; elle illustre surtout le rôle de la cible lente. Quand le latent garde une dispersion non nulle, le modèle conserve au moins une partie de l'information d'entrée. Quand la dispersion tombe vers la zone rouge, le prédicteur peut résoudre l'objectif en produisant une constante.

Ce diagnostic est aussi important qu'une courbe de récompense. Un planner peut parfois obtenir une amélioration courte avec un modèle fragile, mais si le latent collapsait, il n'aurait plus de géométrie exploitable pour comparer des futurs imaginés.

## Mini-test: VICReg punit une représentation constante

Le test suivant encode l'idée de manière directe: un batch constant doit avoir une forte perte de variance, alors qu'un batch aléatoire devrait être moins pénalisé. Ce n'est pas une validation scientifique de VICReg, mais c'est un bon test local de l'implémentation.

Il est important de garder ces tests visibles dans le notebook. Le lecteur voit non seulement l'équation, mais aussi le comportement attendu sur un exemple extrême. C'est la différence entre lire une formule et savoir ce qu'elle protège concrètement.

In [ ]:
# Mini-test : VICReg pénalise un latent constant, pas un latent varié
constant = torch.ones(32, 8)
varied = torch.randn(64, 8) * 2.5
assert variance_loss(constant) > 0.5
assert variance_loss(varied) < 0.1
assert covariance_loss(varied) >= 0
print("VICReg-lite OK", float(variance_loss(constant)), float(variance_loss(varied)))

## Illustration: décorréler les dimensions latentes

La covariance hors diagonale mesure la redondance linéaire entre dimensions. Si toutes les dimensions racontent la même chose, le latent peut avoir une variance non nulle tout en étant pauvre. Le terme de covariance pousse donc le modèle à distribuer l'information sur plusieurs axes.

La figure compare une matrice de covariance après entraînement avec et sans pénalité de covariance. Dans un monde idéal, la diagonale resterait active et les termes hors diagonale seraient plus proches de zéro. Ce n'est pas une exigence absolue, mais c'est une bonne intuition visuelle.

In [ ]:
def covariance_matrix_after_training(use_cov, steps=220):
    torch.manual_seed(0)
    enc = TinyEncoder(latent=8)
    pred = nn.Sequential(nn.Linear(8, 32), nn.SiLU(), nn.Linear(32, 8))
    opt = torch.optim.Adam(list(enc.parameters()) + list(pred.parameters()), lr=1e-3)

    for _ in range(steps):
        x, y = toy_pairs()
        z = enc(x)

        # Invariance aligns the two views; variance/covariance keep the latent informative.
        loss = F.mse_loss(pred(z), enc(y).detach()) + variance_loss(z)
        if use_cov:
            loss = loss + covariance_loss(z)

        opt.zero_grad()
        loss.backward()
        opt.step()

    with torch.no_grad():
        z = enc(toy_pairs(512)[0])
        z = z - z.mean(0, keepdim=True)
        return ((z.T @ z) / (z.shape[0] - 1)).numpy()

fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
matrices = [covariance_matrix_after_training(False), covariance_matrix_after_training(True)]
titles = ["sans covariance", "avec covariance"]
for axis, mat, title in zip(ax, matrices, titles):
    im = axis.imshow(mat, cmap="coolwarm", vmin=-1, vmax=1)
    axis.set_title(title)
fig.colorbar(im, ax=ax.ravel().tolist(), fraction=0.04)
plt.show()

**Lecture.** La version avec covariance ne rend pas instantanément le latent parfait, mais elle réduit la tendance des dimensions à se dupliquer. Dans les grands modèles, cette pression aide à éviter des représentations où quelques axes dominent tout. Dans notre petit notebook, elle sert surtout à rendre visible la notion de redondance.

Ce point prépare la suite: un planner CEM suppose que la distance ou la valeur calculée dans le latent a un sens. Si le latent est dégénéré, le planner optimise un paysage artificiel. Le diagnostic de covariance n'est donc pas décoratif; il touche directement la qualité du contrôle.

# Partie III — Deux régimes d'apprentissage, une architecture commune

Les parties précédentes ont présenté la philosophie JEPA et les mécanismes qui empêchent le collapse. Nous pouvons maintenant poser la question expérimentale au cœur de ce notebook :

> Faut-il d'abord apprendre une représentation prédictive générale, puis y apprendre l'effet des actions, ou faut-il optimiser directement représentation et dynamique pour le contrôle ?

Nous comparons deux réponses :

- **stage-wise** : apprendre d'abord à représenter le monde **sans action**, geler cette représentation, puis apprendre comment les actions la font évoluer ;
- **joint** : apprendre simultanément la représentation et la dynamique action-conditionnée.

Cette comparaison ne doit pas être perturbée par des différences de capacité. Les deux régimes utilisent donc les mêmes dimensions latentes, le même prédicteur résiduel et le même planner CEM/MPC. Seuls changent l'ordre des phases et le chemin suivi par les gradients.

| Brique | Stage-wise | Joint |
|---|---|---|
| Encodeur online | pré-entraîné sans action, puis gelé | entraîné avec la dynamique |
| Teacher EMA | utilisé pendant la phase JEPA | utilisé pendant tout l'entraînement |
| VICReg-lite | régularise le pré-entraînement | régularise continuellement le latent |
| Prédicteur action-conditionné | entraîné après le gel | entraîné avec l'encodeur |
| Reward/continuation | têtes entraînées après le gel | têtes entraînées conjointement |
| CEM/MPC | identique | identique |

Le contrôle de l'architecture est essentiel : si une variante obtient un meilleur résultat, nous voulons l'attribuer à son **régime d'apprentissage**, pas à un réseau plus large ou à un planner différent.

```mermaid
flowchart TB
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    data["Trajectoires réelles<br/>$$~(o_t,a_t,o_{t+1})$$"]

    subgraph staged["Régime stage-wise"]
        pretrain["Phase A — JEPA sans action<br/>Prédiction masquée"]
        freeze["Gel de l'encodeur<br/>$$~E_\star$$"]
        posttrain["Phase B — dynamique actionnée<br/>$$~P_\phi(z_t,a_t)$$"]
    end

    subgraph joint["Régime joint"]
        endtoend["Encodeur + dynamique<br/>optimisés ensemble"]
    end

    planner["Même planner CEM/MPC"]
    env["Contrôle dans l'environnement réel"]

    data --> pretrain --> freeze --> posttrain
    data --> endtoend
    posttrain --> planner
    endtoend --> planner
    planner --> env

    classDef stagedNode fill:none,stroke:#2563eb,stroke-width:2px
    classDef jointNode fill:none,stroke:#d97706,stroke-width:2px
    classDef sharedNode fill:none,stroke:#16a34a,stroke-width:2px
    class pretrain,freeze,posttrain stagedNode
    class endtoend jointNode
    class planner,env sharedNode
```



## Phase A — Apprendre une représentation sans action

La première phase du régime stage-wise ne cherche pas encore à contrôler Pendulum. Elle cherche à construire un espace latent qui capture les régularités temporelles de ses observations.

Une observation vaut :

$$
o_t = [\cos\theta_t,\ \sin\theta_t,\ \dot\theta_t].
$$

Avec seulement trois coordonnées, masquer l'observation entière supprimerait tout contexte, tandis que ne rien masquer rendrait la tâche triviale. Nous construisons donc une tâche intermédiaire :

- le student reçoit $o_t$ complet ;
- il reçoit également une version partiellement masquée $\tilde o_{t+1}$ ;
- un masque binaire $m$ lui indique quelles coordonnées sont visibles ;
- le teacher EMA encode l'observation suivante complète $o_{t+1}$.

Le prédicteur de contexte doit reconstruire la **représentation complète** de l'observation suivante :

$$
\hat z^+_{t+1}
=
Q_\rho\!\left(
E_\theta(o_t),
E_\theta(\tilde o_{t+1}),
m
\right).
$$

La cible est produite par l'encodeur teacher et ne reçoit aucun gradient :

$$
z^+_{t+1}
=
\operatorname{sg}\!\left(E_\xi(o_{t+1})\right).
$$

### Rôle de chaque terme

- $E_\theta$ est l'encodeur **student**, mis à jour par gradient.
- $E_\xi$ est l'encodeur **teacher**, mis à jour uniquement par EMA.
- $Q_\rho$ est le prédicteur de contexte masqué.
- $\tilde o_{t+1}$ contient les coordonnées visibles de l'observation suivante.
- $m$ indique explicitement les coordonnées masquées.
- $\operatorname{sg}$ empêche le gradient de traverser la cible.

### Exemple concret

Supposons :

$$
o_{t+1}=[0.82,\ 0.57,\ 1.40].
$$

Si la vitesse est masquée, le student reçoit :

$$
\tilde o_{t+1}=[0.82,\ 0.57,\ 0],
\qquad
m=[1,\ 1,\ 0].
$$

Il doit alors retrouver la représentation teacher de l'état complet en utilisant :

- l'état précédent $o_t$ ;
- le nouvel angle partiellement visible ;
- les régularités temporelles déjà observées dans les trajectoires.

La prédiction ne se fait pas dans l'espace brut : le réseau ne cherche pas directement à produire `1.40`. Il cherche à prédire le **latent teacher** correspondant à l'observation complète. C'est précisément la signature d'un JEPA.

### Que peut apprendre un modèle sans action ?

L'action reste volontairement cachée pendant cette phase. Le modèle ne peut donc pas apprendre exactement :

> « si j'applique ce couple précis, la vitesse devient celle-ci ».

Il apprend plutôt :

> « parmi les évolutions habituellement observées depuis ce type d'état, quelles structures du futur restent prévisibles ? »

Autrement dit, la Phase A apprend une représentation des régularités du système, mais pas encore une dynamique contrôlable. Elle marginalise implicitement l'action inconnue. C'est une limite assumée et même souhaitée : nous séparons volontairement **comprendre l'évolution du monde** et **comprendre l'effet causal d'une commande**.

La loss de représentation combine l'alignement prédictif et les protections anti-collapse :

$$
\mathcal L_{\mathrm{repr}}
=
\mathcal L_{\mathrm{mask}}
+
\lambda_{\mathrm{var}}\mathcal L_{\mathrm{var}}
+
\lambda_{\mathrm{cov}}\mathcal L_{\mathrm{cov}}.
$$

La perte masquée rapproche la prédiction du teacher. Les termes VICReg-lite vérifient que le latent reste dispersé et que ses dimensions ne deviennent pas redondantes.

Cette phase est notre analogue pédagogique du *mask denoising* de V-JEPA. Les coordonnées vectorielles remplacent les tokens spatio-temporels d'une vidéo. Ce n'est pas une reproduction à l'échelle de V-JEPA2, mais le principe expérimental est conservé : **apprendre une représentation en prédisant une cible latente partiellement cachée, sans reconstruire l'observation brute**.

---

## Transition — Geler la représentation

Après le pré-entraînement, nous conservons l'encodeur student appris et nous le gelons :

$$
E_\star \leftarrow E_\theta,
\qquad
\nabla E_\star = 0.
$$

À partir de cet instant :

- les observations sont toujours projetées dans le même espace latent ;
- les cibles ne bougent plus pendant l'apprentissage de la dynamique ;
- seul le prédicteur action-conditionné apprend à transformer ces latents.

Ce gel joue un rôle conceptuel important. Il empêche la représentation de se déformer pour rendre la loss de dynamique artificiellement facile. Le prédicteur doit s'adapter à la géométrie apprise pendant la Phase A, et non l'inverse.

Le bénéfice attendu est la **stabilité**. Le risque est une perte de **spécialisation** : une représentation utile pour la prédiction masquée n'est pas nécessairement la meilleure métrique pour le contrôle ou pour CEM. Toute la comparaison avec le régime joint repose sur ce compromis.


## Phase B — Apprendre l'effet causal des actions

La seconde phase reçoit enfin les actions. Pour chaque transition réelle :

$$
z_t = E_\star(o_t),
\qquad
z_{t+1}=E_\star(o_{t+1}).
$$

Le prédicteur apprend alors :

$$
\hat z_{t+1}=P_\phi(z_t,a_t).
$$

Contrairement à la Phase A, la question devient explicitement causale :

> Sachant l'état latent courant, comment cette action particulière déplace-t-elle le système ?

Deux actions différentes appliquées au même latent doivent pouvoir produire deux futurs différents. C'est cette propriété qui rend ensuite le modèle utilisable par CEM.

### Pourquoi une dynamique résiduelle ?

Le prédicteur utilise une forme delta :

$$
P_\phi(z_t,a_t)
=
z_t+\Delta_\phi(z_t,a_t).
$$

Le réseau n'a donc pas à reconstruire entièrement le prochain latent. Il apprend seulement son déplacement.

Ce biais correspond bien aux systèmes physiques échantillonnés à haute fréquence : entre deux instants proches, la position et la vitesse évoluent généralement de manière progressive. Le chemin identité stabilise aussi le gradient lorsque la transition est faible.

---

## Teacher forcing et rollout ouvert

Une bonne prédiction à un pas ne garantit pas une bonne planification. CEM applique récursivement le modèle sur ses propres sorties :

$$
\hat z_{t+k}
=
P_\phi(\hat z_{t+k-1},a_{t+k-1}).
$$

Nous utilisons donc deux losses complémentaires.

### Teacher forcing

À chaque pas, le prédicteur repart d'un vrai latent encodé :

$$
\mathcal L_{\mathrm{TF}}
=
\frac{1}{K}
\sum_{k=0}^{K-1}
\left\|
P_\phi(z_{t+k},a_{t+k})
-
z_{t+k+1}
\right\|_1.
$$

Cette loss mesure la qualité locale de la dynamique. Elle est stable, car chaque entrée provient d'une vraie observation.

### Rollout ouvert

Le prédicteur doit aussi apprendre à supporter ses propres petites erreurs :

$$
\mathcal L_{\mathrm{rollout}}
=
\frac{1}{K}
\sum_{k=1}^{K}
\left\|
\hat z_{t+k}-z_{t+k}
\right\|_1.
$$

Ici, seule la première entrée est réelle. Les suivantes sont produites récursivement par le modèle. Cette loss rapproche donc l'entraînement de l'utilisation réelle par CEM.

La loss action-conditionnée devient :

$$
\mathcal L_{\mathrm{AC}}
=
\mathcal L_{\mathrm{TF}}
+
\lambda_{\mathrm{rollout}}\mathcal L_{\mathrm{rollout}}
+
\lambda_r\mathcal L_r
+
\lambda_c\mathcal L_c.
$$

L'ablation présentée plus loin vérifiera si le rollout ouvert apporte réellement un avantage. Nous ne supposons pas qu'il est toujours bénéfique : mal pondéré, il peut exposer trop tôt le modèle à ses propres erreurs et compliquer l'optimisation.


## Extension — Récompense et continuation

Pour Pendulum, le chemin principal reste proche de V-JEPA2-AC : CEM planifie vers un latent-but, sans utiliser une récompense apprise.

Toutes les tâches ne possèdent cependant pas un objectif final unique. HalfCheetah ne cherche pas à atteindre une pose précise : il doit continuer à courir vite tout en limitant l'énergie dépensée. Nous ajoutons donc deux têtes auxiliaires :

$$
(z_t,a_t)
\longmapsto
\left(
\hat z_{t+1},
\hat r_{t+1},
\hat c_{t+1}
\right).
$$

- $\hat r_{t+1}$ estime la récompense immédiate produite par la transition ;
- $\hat c_{t+1}$ estime la probabilité que l'épisode continue ;
- $\hat z_{t+1}$ permet de poursuivre le rollout imaginé.

Les trois sorties sont apprises en parallèle depuis le latent **avant l'action**. Ce choix conserve explicitement l'action dans les têtes de récompense et de continuation, ce qui est important lorsque la récompense contient un coût de contrôle.

Il évite également une cascade inutile :

$$
z_t,a_t
\rightarrow
\hat z_{t+1}
\rightarrow
\hat r_{t+1}.
$$

Dans cette cascade, toute erreur du prédicteur latent contaminerait immédiatement la reward head. Avec des têtes parallèles, la récompense et la continuation peuvent rester localement précises même lorsque le rollout latent dérive légèrement.

Cette extension n'est pas le cœur reward-free de V-JEPA2-AC. Elle constitue notre adaptation aux tâches où l'objectif ne peut pas être décrit par une unique image ou observation-but.


## Le régime joint : mêmes composants, gradients différents

La variante joint utilise le même encodeur et le même prédicteur, mais elle supprime la séparation entre les phases.

À chaque update :

1. l'encodeur produit les latents courants ;
2. le teacher EMA produit les cibles ;
3. le prédicteur action-conditionné prédit les prochains latents ;
4. les losses de dynamique, récompense, continuation et VICReg sont calculées ;
5. le gradient met à jour simultanément l'encodeur et les têtes ;
6. le teacher suit ensuite l'encodeur par EMA.

Le latent peut donc se transformer pour devenir directement utile à la dynamique et au contrôle.

C'est potentiellement un avantage : l'espace peut apprendre à rendre proches les états qui ont des conséquences similaires pour l'action. Mais c'est également une source d'instabilité : le prédicteur poursuit une cible dont la géométrie évolue pendant l'entraînement.

La différence essentielle peut se résumer ainsi :

- **stage-wise** : le monde latent est appris, figé, puis la dynamique s'y adapte ;
- **joint** : le monde latent et la dynamique négocient leur représentation simultanément.

---

## Une variante de contrôle : `stage-wise-fair`

La comparaison principale utilise le même nombre total d'updates :

$$
N_{\mathrm{joint}}
=
N_{\mathrm{repr}}
+
N_{\mathrm{AC}}.
$$

Mais dans le régime joint, le prédicteur reçoit un gradient pendant toutes ces updates, tandis que le prédicteur stage-wise n'existe que pendant la Phase B.

Nous ajoutons donc `stage-wise-fair`, qui garde exactement le même encodeur gelé mais entraîne son prédicteur pendant autant d'updates que celui du régime joint.

Cette variante ne possède pas le même coût total : elle ajoute le pré-entraînement au budget complet du prédicteur. Elle est « équitable » uniquement selon un axe précis, celui du nombre d'updates de dynamique.

Les deux comparaisons répondent ainsi à deux questions différentes :

| Comparaison | Question |
|---|---|
| `joint` vs `stage-wise` | À nombre total d'updates comparable, quelle organisation apprend le mieux ? |
| `joint` vs `stage-wise-fair` | À prédicteur également entraîné, le pré-entraînement puis le gel apportent-ils quelque chose ? |


## Réseaux from scratch

L'implémentation conserve volontairement des réseaux modestes afin que chaque rôle reste identifiable :

- `Encoder` transforme une observation en latent normalisé ;
- `MaskedContextPredictor` n'existe que pendant le pré-entraînement JEPA ;
- `Predictor` apprend la transition action-conditionnée ;
- `RewardHead` prédit la récompense immédiate lorsque la tâche l'exige ;
- `ContinuationHead` indique si les récompenses futures doivent encore compter ;
- `LatentCEMPlanner` recherche les séquences d'actions directement dans le modèle appris.

Le code qui suit ne construit donc pas un unique « gros réseau JEPA ». Il assemble des composants ayant chacun une responsabilité expérimentale précise. Cette séparation nous permettra ensuite de diagnostiquer **où** une variante réussit ou échoue : représentation, effet causal de l'action, drift multi-pas, géométrie du but ou planning.

In [ ]:
def make_mlp(input_dim, hidden_dim, output_dim, n_layers=2):
    layers, dim = [], input_dim
    for _ in range(n_layers):
        layers += [nn.Linear(dim, hidden_dim), nn.SiLU()]
        dim = hidden_dim
    layers.append(nn.Linear(dim, output_dim))
    return nn.Sequential(*layers)


class Encoder(nn.Module):
    def __init__(self, obs_dim, hidden_dim, latent_dim):
        super().__init__()
        self.backbone = make_mlp(obs_dim, hidden_dim, latent_dim)
        self.norm = nn.LayerNorm(latent_dim)

    def forward(self, obs):
        return self.norm(self.backbone(obs))


class MaskedContextPredictor(nn.Module):
    # Prédit le latent complet de o_{t+1} depuis o_t, une vue partielle et son masque.
    def __init__(self, latent_dim, obs_dim, hidden_dim):
        super().__init__()
        self.net = make_mlp(2 * latent_dim + obs_dim, hidden_dim, latent_dim)

    def forward(self, context_latent, partial_latent, visible_mask):
        inputs = torch.cat([context_latent, partial_latent, visible_mask], dim=-1)
        return self.net(inputs)


class Predictor(nn.Module):
    def __init__(self, latent_dim, action_dim, hidden_dim, delta=True):
        super().__init__()
        self.delta = delta
        self.net = make_mlp(latent_dim + action_dim, hidden_dim, latent_dim)

    def forward(self, latent, action):
        update = self.net(torch.cat([latent, action], dim=-1))
        return latent + update if self.delta else update


class RewardHead(nn.Module):
    def __init__(self, latent_dim, action_dim, hidden_dim):
        super().__init__()
        self.net = make_mlp(latent_dim + action_dim, hidden_dim, 1, n_layers=1)

    def forward(self, latent, action):
        return self.net(torch.cat([latent, action], dim=-1)).squeeze(-1)


class ContinuationHead(nn.Module):
    def __init__(self, latent_dim, action_dim, hidden_dim):
        super().__init__()
        self.net = make_mlp(latent_dim + action_dim, hidden_dim, 1, n_layers=1)

    def forward(self, latent, action):
        return self.net(torch.cat([latent, action], dim=-1)).squeeze(-1)

## Assembler les briques dans un agent

Les réseaux précédents ont volontairement été construits séparément : chacun correspond à une
responsabilité précise de l'algorithme. On les rassemble maintenant dans un conteneur léger,
`ActionJEPAAgent`. Il ne cache ni les losses ni les deux protocoles d'entraînement ; il garantit
simplement qu'une variante entraînée possède une interface stable pour encoder, prédire une
transition et planifier.

L'environnement reste hors de la classe. La boucle MPC observe le vrai monde, appelle l'agent,
exécute l'action et recommence. Cette frontière est la même que dans le package : **l'agent connaît
ses modèles ; la boucle connaît l'expérience réelle**.

In [ ]:
@dataclass
class ActionJEPAAgent:
    encoder: nn.Module
    predictor: nn.Module
    reward_head: nn.Module
    continuation_head: nn.Module
    history: dict
    regime: str
    representation_history: dict | None = None

    @torch.no_grad()
    def encode(self, observation):
        observation = torch.as_tensor(observation, dtype=torch.float32)
        return self.encoder(observation)

    def predict_next(self, latent, action):
        return self.predictor(latent, action)

    def predict_transition(self, latent, action):
        # Les trois conséquences de (z_t, a_t) restent des sorties parallèles.
        next_latent = self.predictor(latent, action)
        reward = self.reward_head(latent, action)
        continuation = torch.sigmoid(self.continuation_head(latent, action))
        return next_latent, reward, continuation


# Mini-test : l'agent assemble les briques sans absorber la boucle d'environnement.
_agent_smoke = ActionJEPAAgent(
    Encoder(3, 16, 8), Predictor(8, 1, 16), RewardHead(8, 1, 16),
    ContinuationHead(8, 1, 16), history={}, regime="smoke",
)
_z = _agent_smoke.encode(torch.zeros(2, 3))
_next_z, _reward, _continuation = _agent_smoke.predict_transition(_z, torch.zeros(2, 1))
assert _next_z.shape == (2, 8)
assert _reward.shape == _continuation.shape == (2,)
assert ((_continuation >= 0) & (_continuation <= 1)).all()
print("ActionJEPAAgent : encodeur, dynamique et têtes auxiliaires assemblés")

**Mini-test.** Les deux prédicteurs doivent conserver la dimension latente, tout en recevant des informations différentes.

In [ ]:
torch.manual_seed(0)
enc_test = Encoder(3, 64, 16)
mask_pred_test = MaskedContextPredictor(16, 3, 64)
dyn_test = Predictor(16, 1, 64)
rew_test = RewardHead(16, 1, 64)
con_test = ContinuationHead(16, 1, 64)

obs_test = torch.randn(8, 3)
z_test = enc_test(obs_test)
mask_test = torch.tensor([[1.0, 0.0, 1.0]]).expand(8, -1)
masked_prediction = mask_pred_test(z_test, z_test, mask_test)
action_test = torch.randn(8, 1)
next_z_test = dyn_test(z_test, action_test)

assert z_test.shape == masked_prediction.shape == next_z_test.shape == (8, 16)
assert rew_test(z_test, action_test).shape == con_test(z_test, action_test).shape == (8,)
assert torch.allclose(next_z_test, z_test + dyn_test.net(torch.cat([z_test, action_test], -1)))
print("réseaux OK :", z_test.shape, "| sorties transition :", rew_test(z_test, action_test).shape)

## EMA : une cible lente réservée à la représentation

Pendant la phase JEPA — et seulement pendant cette phase dans la variante stage-wise — le teacher
suit le student par $\xi\leftarrow\tau\xi+(1-\tau)\theta$. Après le gel de $E_\star$, aucune EMA
n'est nécessaire : la cible action-conditionnée est calculée dans un espace devenu fixe.

In [ ]:
@torch.no_grad()
def ema_update(target, online, tau):
    for target_param, online_param in zip(target.parameters(), online.parameters()):
        target_param.mul_(tau).add_(online_param.detach(), alpha=1.0 - tau)
    for target_buffer, online_buffer in zip(target.buffers(), online.buffers()):
        target_buffer.copy_(online_buffer)


online_test, target_test = nn.Linear(4, 4), nn.Linear(4, 4)
with torch.no_grad():
    for parameter in online_test.parameters(): parameter.fill_(1.0)
    for parameter in target_test.parameters(): parameter.zero_()
ema_update(target_test, online_test, tau=0.5)
assert torch.allclose(next(target_test.parameters()), torch.full_like(next(target_test.parameters()), 0.5))
print("EMA OK : la cible a parcouru exactement la moitié du chemin")

# Partie IV — Buffer de séquences et sémantique terminale

Les rollouts utilisent des fenêtres contiguës. Le buffer doit aussi distinguer deux événements :

- `episode_end = terminated or truncated` ferme la séquence stockée ;
- `terminal = terminated` devient la cible de continuation.

Confondre les deux apprendrait à Pendulum qu'une limite administrative de 200 pas est une catastrophe
physique, alors que le temps restant n'apparaît même pas dans l'observation.

In [ ]:
class SequenceBuffer:
    def __init__(self, capacity, seed=0):
        self.capacity = int(capacity)
        self.episodes, self.current = [], []
        self.rng = np.random.default_rng(seed)

    def reset_rng(self, seed):
        self.rng = np.random.default_rng(seed)

    def add(self, obs, action, reward, next_obs, terminal, episode_end):
        self.current.append((
            np.asarray(obs, np.float32), np.asarray(action, np.float32), float(reward),
            np.asarray(next_obs, np.float32), bool(terminal),
        ))
        if episode_end:
            self.flush()

    def flush(self):
        if not self.current:
            return
        obs, action, reward, next_obs, terminal = zip(*self.current)
        self.episodes.append({
            "obs": np.stack(obs), "action": np.stack(action),
            "reward": np.asarray(reward, np.float32), "next_obs": np.stack(next_obs),
            "done": np.asarray(terminal, np.float32),
        })
        self.current = []
        while len(self) > self.capacity and len(self.episodes) > 1:
            self.episodes.pop(0)

    def __len__(self):
        return sum(len(episode["obs"]) for episode in self.episodes) + len(self.current)

    def sample(self, batch_size, rollout_len):
        valid = [episode for episode in self.episodes if len(episode["obs"]) >= rollout_len]
        if not valid:
            raise RuntimeError("aucun épisode assez long")
        batches = {key: [] for key in ["obs", "action", "reward", "done"]}
        for _ in range(batch_size):
            episode = valid[int(self.rng.integers(len(valid)))]
            start = int(self.rng.integers(0, len(episode["obs"]) - rollout_len + 1))
            stop = start + rollout_len
            # K transitions donnent K+1 observations : o_t puis les K next_obs réels.
            obs_sequence = np.concatenate([
                episode["obs"][start:start + 1], episode["next_obs"][start:stop]
            ], axis=0)
            batches["obs"].append(obs_sequence)
            batches["action"].append(episode["action"][start:stop])
            batches["reward"].append(episode["reward"][start:stop])
            batches["done"].append(episode["done"][start:stop])
        return {key: torch.tensor(np.stack(value), dtype=torch.float32) for key, value in batches.items()}

    def observations(self):
        arrays = [np.concatenate([ep["obs"], ep["next_obs"][-1:]], axis=0) for ep in self.episodes]
        return np.concatenate(arrays, axis=0).astype(np.float32)

**Mini-test.** Une troncature ferme l'épisode sans créer de cible terminale, et une fenêtre de $K$ transitions contient bien $K+1$ observations.

In [ ]:
buffer_test = SequenceBuffer(100, seed=0)
for step in range(5):
    episode_end = step == 4
    buffer_test.add(
        np.array([step], np.float32), np.array([0.0], np.float32), 1.0,
        np.array([step + 1], np.float32), terminal=False, episode_end=episode_end,
    )
batch_test = buffer_test.sample(batch_size=3, rollout_len=3)
assert batch_test["obs"].shape == (3, 4, 1)
assert batch_test["action"].shape == (3, 3, 1)
assert float(batch_test["done"].sum()) == 0.0
assert len(buffer_test.episodes) == 1
print("buffer OK : troncature séparée du terminal", {k: tuple(v.shape) for k, v in batch_test.items()})

# Partie V — Données et protocoles comparables

Comparer deux régimes d'apprentissage exige de contrôler ce qu'ils ont réellement eu le droit de voir. Si `joint` recevait davantage de transitions, ou si `stage-wise` était évalué sur des épisodes déjà utilisés pendant son pré-entraînement, une différence de score serait impossible à interpréter correctement.

Nous construisons donc **une seule fois** deux jeux d'épisodes exploratoires :

- un ensemble d'entraînement, utilisé pour optimiser les modèles ;
- un ensemble de validation, collecté avec d'autres seeds et jamais utilisé par les optimiseurs.

La séparation se fait au niveau des **épisodes complets**, avant de construire les fenêtres temporelles. Deux fenêtres issues du même épisode sont fortement corrélées : placer l'une dans le train et l'autre dans la validation provoquerait une fuite de données discrète. Le modèle serait évalué sur une trajectoire qu'il connaît déjà presque entièrement.

Toutes les variantes utilisent ensuite exactement :

- les mêmes observations ;
- les mêmes actions aléatoires ;
- les mêmes récompenses et terminaisons ;
- les mêmes longueurs de rollout ;
- les mêmes épisodes de validation ;
- les mêmes resets lors de l'évaluation finale.

Cette première contrainte est appelée **data matching** :

$$
\mathcal D_{\mathrm{joint}}
=
\mathcal D_{\mathrm{stage}}
=
\mathcal D_{\mathrm{random\text{-}frozen}}.
$$

Ainsi, si une variante apprend mieux, ce n'est pas parce qu'elle a rencontré une trajectoire plus favorable ou davantage d'expérience réelle.

## Contrôler aussi le budget d'optimisation

L'égalité des données ne suffit pas. Le régime `stage-wise` possède deux phases successives :

$$
N_{\mathrm{stage}}
=
N_{\mathrm{repr}}
+
N_{\mathrm{AC}},
$$

où $N_{\mathrm{repr}}$ désigne les updates du pré-entraînement JEPA et $N_{\mathrm{AC}}$ celles du prédicteur action-conditionné.

La variante `joint` est donc entraînée pendant le même nombre total d'updates :

$$
N_{\mathrm{joint}}
=
N_{\mathrm{repr}}
+
N_{\mathrm{AC}}.
$$

Cette comparaison est **update-count matched** : les deux méthodes disposent du même nombre d'étapes d'optimisation. Elle répond à la question pratique :

> Avec un budget global fixé, vaut-il mieux consacrer une partie des updates à un pré-entraînement séparé ou optimiser directement toute l'architecture pour la dynamique ?

Il faut néanmoins être précis : une update n'a pas exactement le même coût dans les deux régimes.

- Pendant la Phase A, `stage-wise` met à jour l'encodeur et le prédicteur masqué.
- Pendant la Phase B, il met seulement à jour le prédicteur action-conditionné et les têtes auxiliaires.
- Dans `joint`, chaque update traverse simultanément l'encodeur, le prédicteur et les têtes.

Le nombre d'updates est donc un **proxy simple et reproductible du compute**, pas une égalité exacte de FLOPs, de temps GPU ou de gradients calculés.

## Pourquoi ajouter `stage-wise-fair` ?

Une seconde asymétrie demeure. Dans le régime joint, le prédicteur action-conditionné existe pendant toutes les updates :

$$
N_{\mathrm{pred,joint}}
=
N_{\mathrm{repr}}+N_{\mathrm{AC}}.
$$

Dans le stage-wise classique, il n'est entraîné qu'après le gel de l'encodeur :

$$
N_{\mathrm{pred,stage}}
=
N_{\mathrm{AC}}.
$$

Si `joint` obtient un meilleur score, deux explications restent donc possibles :

1. son encodeur s'est spécialisé conjointement pour le contrôle ;
2. son prédicteur a simplement reçu davantage d'updates.

La variante `stage-wise-fair` isole ces deux effets. Elle réutilise le même encodeur pré-entraîné et gelé que `stage-wise`, mais entraîne son prédicteur pendant autant d'updates que celui de `joint` :

$$
N_{\mathrm{pred,stage\text{-}fair}}
=
N_{\mathrm{pred,joint}}.
$$

Son budget total est alors plus élevé :

$$
N_{\mathrm{stage\text{-}fair}}
=
N_{\mathrm{repr}}
+
N_{\mathrm{pred,joint}}.
$$

Elle n'est donc pas « équitable » sur le compute total. Elle l'est uniquement sur le **budget du prédicteur**. Le suffixe `fair` désigne ici une comparaison *dynamics-matched*, pas une égalité universelle de ressources.

## Les deux comparaisons principales

| Comparaison | Élément apparié | Question |
|---|---|---|
| `joint` vs `stage-wise` | données et nombre total d'updates | Quelle organisation utilise le mieux un budget global fixé ? |
| `joint` vs `stage-wise-fair` | données et updates du prédicteur | Le pré-entraînement gelé reste-t-il compétitif lorsque la dynamique reçoit autant d'optimisation ? |
| `stage-wise` vs `random-frozen` | données et updates action-conditionnées | La représentation JEPA apporte-t-elle davantage qu'une projection aléatoire fixe ? |

`random-frozen` est important : son encodeur reste aléatoire, mais son prédicteur est réellement entraîné. Il mesure donc ce que l'on peut déjà obtenir grâce à une projection de grande dimension et à CEM, sans apprentissage de représentation.

Enfin, toutes les variantes sont évaluées avec les mêmes seeds et les mêmes états initiaux. Cette évaluation **appariée** réduit la variance environnementale : deux méthodes affrontent exactement les mêmes configurations difficiles ou faciles.

Le protocole cherche ainsi à séparer quatre causes souvent confondues :

```text
qualité des données
≠ quantité d'optimisation
≠ qualité de la représentation
≠ qualité du prédicteur
```

Sans ces contrôles, comparer seulement les retours finaux donnerait un classement, mais aucune explication scientifique de ce classement.

In [ ]:
# Configuration du dataset et des expériences comparées.
SEED = 0
ENV_ID = "Pendulum-v1"

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


In [ ]:
@dataclass
class RepresentationConfig:
    latent_dim: int = 24
    hidden_dim: int = 128
    batch_size: int = 96
    learning_rate: float = 3e-4
    ema_tau: float = 0.99
    mask_fraction: float = 0.34
    noise_std: float = 0.01
    variance_weight: float = 1.0
    covariance_weight: float = 0.04
    target_std: float = 1.0
    updates: int = 1200


@dataclass
class ActionConfig:
    rollout_len: int = 5
    batch_size: int = 96
    learning_rate: float = 3e-4
    reward_weight: float = 0.25
    continuation_weight: float = 0.05
    grad_clip: float = 10.0
    ac_updates: int = 1800


rep_cfg = RepresentationConfig()
ac_cfg = ActionConfig()

In [ ]:
def collect_random_episodes(env, buffer, n_episodes, max_steps=200, seed=0):
    returns = []
    for episode in range(n_episodes):
        episode_seed = seed + episode
        obs, _ = env.reset(seed=episode_seed)
        env.action_space.seed(episode_seed)
        total = 0.0
        for _ in range(max_steps):
            action = env.action_space.sample()
            next_obs, reward, terminated, truncated, _ = env.step(action)
            episode_end = bool(terminated or truncated)
            buffer.add(obs, action, reward, next_obs, terminal=bool(terminated), episode_end=episode_end)
            total += float(reward)
            obs = next_obs
            if episode_end:
                break
        if buffer.current:
            buffer.flush()
        returns.append(total)
    return np.asarray(returns)


train_buffer = SequenceBuffer(50_000, seed=SEED)
val_buffer = SequenceBuffer(10_000, seed=SEED + 1)
train_random_returns = collect_random_episodes(env, train_buffer, n_episodes=18, seed=10)
val_random_returns = collect_random_episodes(env, val_buffer, n_episodes=5, seed=10_000)

print(f"train: {len(train_buffer)} transitions / {len(train_buffer.episodes)} épisodes")
print(f"validation: {len(val_buffer)} transitions / {len(val_buffer.episodes)} épisodes")
print(f"retour random train: {train_random_returns.mean():.1f} ± {train_random_returns.std():.1f}")

## Phase A — pré-entraînement JEPA masqué

Pour chaque paire contiguë $(o_t,o_{t+1})$, on masque au moins une coordonnée de $o_{t+1}$. Le
student dispose de $o_t$, de la vue partielle et du masque ; le teacher reçoit la cible complète.
La loss L1 joue le rôle d'alignement prédictif, tandis que VICReg-lite protège la représentation.

In [ ]:
def effective_rank(latents, eps=1e-8):
    centered = latents - latents.mean(0, keepdim=True)
    singular_values = torch.linalg.svdvals(centered)
    probabilities = singular_values.square()
    probabilities = probabilities / probabilities.sum().clamp_min(eps)
    entropy = -(probabilities * probabilities.clamp_min(eps).log()).sum()
    return float(entropy.exp().detach())


def make_partial_view(target_obs, mask_fraction, noise_std, generator):
    batch_size, obs_dim_local = target_obs.shape
    n_masked = min(obs_dim_local - 1, max(1, round(mask_fraction * obs_dim_local)))
    visible = torch.ones_like(target_obs)
    for row in range(batch_size):
        indices = torch.randperm(obs_dim_local, generator=generator)[:n_masked]
        visible[row, indices] = 0.0
    noise = torch.randn(target_obs.shape, generator=generator) * noise_std
    return (target_obs + noise) * visible, visible


def representation_step(model, batch, config, generator, update):
    context_obs, target_obs = batch["obs"][:, 0], batch["obs"][:, 1]
    partial_obs, visible_mask = make_partial_view(
        target_obs, config.mask_fraction, config.noise_std, generator
    )
    context_z = model["online"](context_obs)
    partial_z = model["online"](partial_obs)
    prediction = model["mask_predictor"](context_z, partial_z, visible_mask)
    with torch.no_grad():
        target_z = model["teacher"](target_obs)

    prediction_loss = F.l1_loss(prediction, target_z)
    var_loss = variance_loss(context_z, config.target_std)
    cov_loss = covariance_loss(context_z)
    total = prediction_loss + config.variance_weight * var_loss + config.covariance_weight * cov_loss

    if update:
        model["optimizer"].zero_grad()
        total.backward()
        nn.utils.clip_grad_norm_(
            list(model["online"].parameters()) + list(model["mask_predictor"].parameters()), 10.0
        )
        model["optimizer"].step()
        ema_update(model["teacher"], model["online"], config.ema_tau)

    return {
        "prediction": float(prediction_loss.detach()),
        "variance": float(var_loss.detach()),
        "covariance": float(cov_loss.detach()),
        "latent_std": latent_std_value(context_z.detach()),
        "effective_rank": effective_rank(context_z.detach()),
    }


def train_representation(train_data, val_data, obs_dim_local, config, seed, verbose=False):
    torch.manual_seed(seed)
    online = Encoder(obs_dim_local, config.hidden_dim, config.latent_dim)
    teacher = copy.deepcopy(online)
    for parameter in teacher.parameters():
        parameter.requires_grad_(False)
    mask_predictor = MaskedContextPredictor(config.latent_dim, obs_dim_local, config.hidden_dim)
    optimizer = torch.optim.Adam(
        list(online.parameters()) + list(mask_predictor.parameters()), lr=config.learning_rate
    )
    model = {"online": online, "teacher": teacher, "mask_predictor": mask_predictor, "optimizer": optimizer}
    history = {key: [] for key in ["prediction", "val_prediction", "variance", "covariance", "latent_std", "effective_rank"]}
    generator = torch.Generator().manual_seed(seed + 100)
    iterator = trange(config.updates, desc="JEPA masqué", disable=not verbose)
    train_data.reset_rng(seed + 200)
    for step in iterator:
        metrics = representation_step(model, train_data.sample(config.batch_size, 1), config, generator, True)
        for key in ["prediction", "variance", "covariance", "latent_std", "effective_rank"]:
            history[key].append(metrics[key])
        if step % 10 == 0:
            val_generator = torch.Generator().manual_seed(999)
            with torch.no_grad():
                val_metrics = representation_step(
                    model, val_data.sample(config.batch_size, 1), config, val_generator, False
                )
            history["val_prediction"].append((step, val_metrics["prediction"]))
    frozen = copy.deepcopy(online).eval()
    for parameter in frozen.parameters():
        parameter.requires_grad_(False)
    return {"encoder": frozen, "online": online, "teacher": teacher, "mask_predictor": mask_predictor, "history": history}

**Vérification.** Le student doit recevoir une vue réellement partielle, le teacher ne doit avoir aucun gradient et l'EMA doit être la seule mise à jour de sa branche.

In [ ]:
partial_demo, visible_demo = make_partial_view(
    torch.tensor([[1.0, 0.5, -0.2], [0.1, -0.8, 0.4]]),
    rep_cfg.mask_fraction, 0.0, torch.Generator().manual_seed(0),
)
assert (visible_demo == 0).sum(dim=1).min() >= 1
assert (visible_demo == 1).sum(dim=1).min() >= 1
print("cible complète :\n", np.round(np.array([[1.0, 0.5, -0.2], [0.1, -0.8, 0.4]]), 2))
print("vue student    :\n", partial_demo.numpy())
print("masque visible :\n", visible_demo.numpy())

## Phase B — post-entraînement action-conditionné

L'encodeur est maintenant figé. Les cibles $z_{t+k}=E_\star(o_{t+k})$ ne bougent plus. Le
prédicteur optimise à la fois une loss teacher-forcing et une loss de rollout ouvert :

$$
\mathcal L_{AC}=\mathcal L_{TF}+\mathcal L_{rollout}
+\lambda_r\mathcal L_r+\lambda_c\mathcal L_c.
$$

Les têtes sont entraînées sur les vrais latents pré-action. Elles restent auxiliaires sur Pendulum ;
le contrôle principal utilisera uniquement la distance au latent-but.

In [ ]:
def action_losses(encoder, predictor, reward_head, continuation_head, batch, config, mode="tf_rollout"):
    obs, actions = batch["obs"], batch["action"]
    rewards, terminals = batch["reward"], batch["done"]
    batch_size, sequence_len, obs_size = obs.shape
    rollout_len = sequence_len - 1
    with torch.no_grad():
        latents = encoder(obs.reshape(-1, obs_size)).reshape(batch_size, sequence_len, -1)

    true_current = latents[:, :-1]
    target_next = latents[:, 1:]
    flat_current = true_current.reshape(-1, true_current.shape[-1])
    flat_actions = actions.reshape(-1, actions.shape[-1])
    teacher_forced = predictor(flat_current, flat_actions).reshape_as(target_next)
    tf_loss = F.l1_loss(teacher_forced, target_next)

    imagined, rollout_latent = [], latents[:, 0]
    for step in range(rollout_len):
        rollout_latent = predictor(rollout_latent, actions[:, step])
        imagined.append(rollout_latent)
    imagined = torch.stack(imagined, dim=1)
    rollout_loss = F.l1_loss(imagined, target_next)

    reward_prediction = reward_head(flat_current, flat_actions).reshape_as(rewards)
    continuation_logits = continuation_head(flat_current, flat_actions).reshape_as(terminals)
    reward_loss = F.mse_loss(reward_prediction, rewards)
    continuation_loss = F.binary_cross_entropy_with_logits(continuation_logits, 1.0 - terminals)
    latent_loss = tf_loss if mode == "tf" else tf_loss + rollout_loss
    total = latent_loss + config.reward_weight * reward_loss + config.continuation_weight * continuation_loss
    return total, {
        "tf": float(tf_loss.detach()), "rollout": float(rollout_loss.detach()),
        "reward": float(reward_loss.detach()), "continuation": float(continuation_loss.detach()),
    }


def train_action_model(frozen_encoder, train_data, val_data, action_dim_local, config, seed, mode="tf_rollout", verbose=False):
    torch.manual_seed(seed)
    latent_dim = int(frozen_encoder.norm.normalized_shape[0])
    predictor = Predictor(latent_dim, action_dim_local, rep_cfg.hidden_dim)
    reward_head = RewardHead(latent_dim, action_dim_local, rep_cfg.hidden_dim)
    continuation_head = ContinuationHead(latent_dim, action_dim_local, rep_cfg.hidden_dim)
    parameters = list(predictor.parameters()) + list(reward_head.parameters()) + list(continuation_head.parameters())
    optimizer = torch.optim.Adam(parameters, lr=config.learning_rate)
    encoder_snapshot = [parameter.detach().clone() for parameter in frozen_encoder.parameters()]
    history = {key: [] for key in ["tf", "rollout", "reward", "continuation", "val_rollout"]}
    train_data.reset_rng(seed + 300)
    iterator = trange(config.ac_updates, desc=f"AC {mode}", disable=not verbose)
    for step in iterator:
        batch = train_data.sample(config.batch_size, config.rollout_len)
        total, metrics = action_losses(
            frozen_encoder, predictor, reward_head, continuation_head, batch, config, mode
        )
        optimizer.zero_grad(); total.backward()
        nn.utils.clip_grad_norm_(parameters, config.grad_clip); optimizer.step()
        for key in ["tf", "rollout", "reward", "continuation"]:
            history[key].append(metrics[key])
        if step % 10 == 0:
            with torch.no_grad():
                _, val_metrics = action_losses(
                    frozen_encoder, predictor, reward_head, continuation_head,
                    val_data.sample(config.batch_size, config.rollout_len), config, mode,
                )
            history["val_rollout"].append((step, val_metrics["rollout"]))
    assert all(torch.equal(before, after) for before, after in zip(encoder_snapshot, frozen_encoder.parameters()))
    return ActionJEPAAgent(
        encoder=frozen_encoder,
        predictor=predictor,
        reward_head=reward_head,
        continuation_head=continuation_head,
        history=history,
        regime="stage-wise",
    )

## Variante jointe — représentation et dynamique apprennent ensemble

La variante jointe reprend l'ancienne architecture du notebook, mais avec les mêmes losses L1
teacher-forcing/rollout et le même budget total d'updates. Le teacher EMA et VICReg restent actifs
pendant toute l'optimisation parce que l'encodeur continue de bouger.

In [ ]:
def train_joint_model(train_data, val_data, obs_dim_local, action_dim_local, rep_config, action_config, seed, verbose=False):
    torch.manual_seed(seed)
    online = Encoder(obs_dim_local, rep_config.hidden_dim, rep_config.latent_dim)
    teacher = copy.deepcopy(online)
    for parameter in teacher.parameters(): parameter.requires_grad_(False)
    predictor = Predictor(rep_config.latent_dim, action_dim_local, rep_config.hidden_dim)
    reward_head = RewardHead(rep_config.latent_dim, action_dim_local, rep_config.hidden_dim)
    continuation_head = ContinuationHead(rep_config.latent_dim, action_dim_local, rep_config.hidden_dim)
    parameters = list(online.parameters()) + list(predictor.parameters()) + list(reward_head.parameters()) + list(continuation_head.parameters())
    optimizer = torch.optim.Adam(parameters, lr=action_config.learning_rate)
    updates = rep_config.updates + action_config.ac_updates
    history = {key: [] for key in ["tf", "rollout", "reward", "continuation", "latent_std", "effective_rank", "val_rollout"]}
    train_data.reset_rng(seed + 400)
    iterator = trange(updates, desc="entraînement joint", disable=not verbose)
    for step in iterator:
        batch = train_data.sample(action_config.batch_size, action_config.rollout_len)
        obs, actions = batch["obs"], batch["action"]
        rewards, terminals = batch["reward"], batch["done"]
        B, Kp1, O = obs.shape; K = Kp1 - 1
        online_latents = online(obs.reshape(-1, O)).reshape(B, Kp1, rep_config.latent_dim)
        with torch.no_grad():
            target_latents = teacher(obs.reshape(-1, O)).reshape(B, Kp1, rep_config.latent_dim)
        current = online_latents[:, :-1]
        flat_current = current.reshape(-1, rep_config.latent_dim)
        flat_actions = actions.reshape(-1, action_dim_local)
        tf_pred = predictor(flat_current, flat_actions).reshape(B, K, rep_config.latent_dim)
        tf_loss = F.l1_loss(tf_pred, target_latents[:, 1:])
        z, imagined = online_latents[:, 0], []
        for k in range(K):
            z = predictor(z, actions[:, k]); imagined.append(z)
        rollout_loss = F.l1_loss(torch.stack(imagined, 1), target_latents[:, 1:])
        reward_loss = F.mse_loss(reward_head(flat_current, flat_actions).reshape_as(rewards), rewards)
        cont_loss = F.binary_cross_entropy_with_logits(
            continuation_head(flat_current, flat_actions).reshape_as(terminals), 1.0 - terminals
        )
        flat = online_latents.reshape(-1, rep_config.latent_dim)
        var_loss = variance_loss(flat, rep_config.target_std); cov_loss = covariance_loss(flat)
        total = tf_loss + rollout_loss + action_config.reward_weight * reward_loss + action_config.continuation_weight * cont_loss
        total = total + rep_config.variance_weight * var_loss + rep_config.covariance_weight * cov_loss
        optimizer.zero_grad(); total.backward()
        nn.utils.clip_grad_norm_(parameters, action_config.grad_clip); optimizer.step()
        ema_update(teacher, online, rep_config.ema_tau)
        history["tf"].append(float(tf_loss.detach())); history["rollout"].append(float(rollout_loss.detach()))
        history["reward"].append(float(reward_loss.detach())); history["continuation"].append(float(cont_loss.detach()))
        history["latent_std"].append(latent_std_value(flat.detach())); history["effective_rank"].append(effective_rank(flat.detach()))
        if step % 10 == 0:
            val_batch = val_data.sample(action_config.batch_size, action_config.rollout_len)
            with torch.no_grad():
                val_obs, val_actions = val_batch["obs"], val_batch["action"]
                VB, VKp1, VO = val_obs.shape; VK = VKp1 - 1
                val_online = online(val_obs.reshape(-1, VO)).reshape(VB, VKp1, -1)
                val_target = teacher(val_obs.reshape(-1, VO)).reshape(VB, VKp1, -1)
                vz, preds = val_online[:, 0], []
                for k in range(VK): vz = predictor(vz, val_actions[:, k]); preds.append(vz)
                history["val_rollout"].append((step, float(F.l1_loss(torch.stack(preds, 1), val_target[:, 1:]))))
    frozen = copy.deepcopy(online).eval()
    for parameter in frozen.parameters(): parameter.requires_grad_(False)
    return ActionJEPAAgent(
        encoder=frozen,
        predictor=predictor,
        reward_head=reward_head,
        continuation_head=continuation_head,
        history=history,
        regime="joint",
    )

## Entraîner les variantes avec le même budget

On compare **quatre régimes** sur **les mêmes transitions réelles**, avec des budgets d'optimisation **explicitement contrôlés** :

- **stage-wise** : Phase A (`updates`) puis Phase B (`ac_updates`) sur l'encodeur gelé. Total = `updates + ac_updates`.
- **joint** : encodeur + dynamique entraînés ensemble pendant `updates + ac_updates` pas.
- **stage-wise-fair** : *même* encodeur pré-entraîné gelé, mais le prédicteur AC reçoit **autant de pas que le prédicteur joint** (`updates + ac_updates`).
- **random-frozen** : encodeur aléatoire gelé + prédicteur AC (`ac_updates`), pour isoler l'apport de la représentation.

Deux comparaisons en découlent — et il faut **les deux** pour conclure honnêtement :

| Comparaison | Ce qui est apparié | Question posée |
|---|---|---|
| **joint vs stage-wise** | le **nombre total d'updates** | à budget d'optimisation comparable, vaut-il mieux tout apprendre ensemble ? |
| **joint vs stage-wise-fair** | les **pas du prédicteur** | à dynamique également entraînée, une représentation pré-entraînée puis gelée aide-t-elle ? |

Sans la variante équitable, le prédicteur *joint* serait nettement plus entraîné que celui de *stage-wise* : la comparaison mélangerait alors « effet de la représentation » et « effet d'un prédicteur mieux optimisé », et ne montrerait rien de net.

In [ ]:
TRAIN_SEEDS = [0, 1, 2, 3, 4]
fair_ac_cfg = replace(ac_cfg, ac_updates=rep_cfg.updates + ac_cfg.ac_updates)
experiments = {}
for seed in TRAIN_SEEDS:
    representation = train_representation(
        train_buffer, val_buffer, obs_dim, rep_cfg, seed, verbose=(seed == 0)
    )
    stagewise = train_action_model(
        representation["encoder"], train_buffer, val_buffer, action_dim,
        ac_cfg, seed, mode="tf_rollout", verbose=(seed == 0),
    )
    stagewise.representation_history = representation["history"]

    # Variante ÉQUITABLE : même encodeur gelé, mais prédicteur entraîné autant que le joint.
    stagewise_fair = train_action_model(
        representation["encoder"], train_buffer, val_buffer, action_dim,
        fair_ac_cfg, seed, mode="tf_rollout",
    )
    stagewise_fair.regime = "stage-wise-fair"

    torch.manual_seed(seed)
    random_encoder = Encoder(obs_dim, rep_cfg.hidden_dim, rep_cfg.latent_dim).eval()
    for parameter in random_encoder.parameters(): parameter.requires_grad_(False)
    random_frozen = train_action_model(
        random_encoder, train_buffer, val_buffer, action_dim,
        ac_cfg, seed, mode="tf_rollout",
    )
    random_frozen.regime = "random-frozen"

    joint = train_joint_model(
        train_buffer, val_buffer, obs_dim, action_dim, rep_cfg, ac_cfg, seed, verbose=(seed == 0)
    )
    experiments[seed] = {"stage-wise": stagewise, "stage-wise-fair": stagewise_fair,
                         "joint": joint, "random-frozen": random_frozen}

print("modèles entraînés :", {seed: list(models) for seed, models in experiments.items()})
print(f"budgets prédicteur — stage-wise: {ac_cfg.ac_updates} | stage-wise-fair: {fair_ac_cfg.ac_updates} "
      f"| joint: {rep_cfg.updates + ac_cfg.ac_updates}")

## Diagnostics d'apprentissage séparés

In [ ]:
seed0 = experiments[0]
rep_history = seed0["stage-wise"].representation_history

fig, axes = plt.subplots(2, 3, figsize=(13, 6))

# ---- Phase A : représentation masquée ----
axes[0, 0].plot(smooth(rep_history["prediction"]), label="train")
val_steps, val_values = zip(*rep_history["val_prediction"])
axes[0, 0].plot(val_steps, val_values, label="validation")
axes[0, 0].set_title("Phase A — prédiction masquée L1")
axes[0, 0].set_xlabel("update")
axes[0, 0].set_ylabel("loss")
axes[0, 0].legend()

axes[0, 1].plot(smooth(rep_history["latent_std"]))
axes[0, 1].set_title("Phase A — latent_std")
axes[0, 1].set_xlabel("update")
axes[0, 1].set_ylabel("écart-type moyen")

axes[0, 2].plot(smooth(rep_history["effective_rank"]))
axes[0, 2].set_title("Phase A — rang effectif")
axes[0, 2].set_xlabel("update")
axes[0, 2].set_ylabel("rang effectif")

# ---- Phase B : dynamique action-conditionnée ----
model_colors = [
    ("stage-wise", "#2563eb"),
    ("stage-wise-fair", "#0ea5e9"),
    ("joint", "#d97706"),
    ("random-frozen", "#64748b"),
]

for name, color in model_colors:
    hist = seed0[name].history

    axes[1, 0].plot(smooth(hist["tf"]), label=name, color=color)
    axes[1, 1].plot(smooth(hist["rollout"]), label=name, color=color)

    if hist.get("val_rollout"):
        val_s, val_v = zip(*hist["val_rollout"])
        axes[1, 2].plot(val_s, val_v, label=name, color=color)

axes[1, 0].set_title("Phase B — teacher forcing")
axes[1, 0].set_xlabel("update")
axes[1, 0].set_ylabel("loss")

axes[1, 1].set_title("Phase B — rollout train")
axes[1, 1].set_xlabel("update")
axes[1, 1].set_ylabel("loss")

axes[1, 2].set_title("Phase B — rollout validation")
axes[1, 2].set_xlabel("update")
axes[1, 2].set_ylabel("loss tenue à part")

for axis in axes[1]:
    axis.legend(fontsize=8)

fig.tight_layout()
plt.show()

## Comment lire ces six courbes ?

La première ligne évalue la **représentation** apprise pendant la Phase A. La seconde évalue la **dynamique action-conditionnée** pendant la Phase B. Ces deux lignes ne répondent donc pas à la même question :

- Phase A : *le latent apprend-il une structure prédictive sans s'effondrer ?*
- Phase B : *peut-on prédire son évolution lorsque l'action est connue ?*

Une faible loss n'est jamais suffisante seule. Elle doit être lue avec la validation, la dispersion du latent et son rang effectif.

### Phase A — Loss de prédiction masquée

Le premier panneau trace la distance L1 entre le latent prédit par le student et la cible produite par le teacher :

$$
\mathcal L_{\mathrm{mask}}
=
\left\|
Q_\rho\!\left(E_\theta(o_t),E_\theta(\tilde o_{t+1}),m\right)
-
\operatorname{sg}\!\left(E_\xi(o_{t+1})\right)
\right\|_1.
$$

La loss train et la loss validation passent ici d'environ `0.8` à moins de `0.1`. Le modèle apprend donc réellement à compléter la représentation de l'observation suivante.

Le fait important est que les deux courbes restent très proches :

- si la loss train baissait mais pas la validation, le modèle mémoriserait les trajectoires exploratoires ;
- si les deux restaient hautes, la tâche masquée serait trop difficile ou le modèle insuffisant ;
- si elles baissaient presque instantanément vers zéro, il faudrait soupçonner un collapse.

Ici, la validation suit le train sans écart croissant : le mécanisme appris se transfère aux épisodes tenus à part.

La courbe atteint cependant un plateau après quelques centaines d'updates. Les updates suivantes modifient peu la loss de prédiction brute. Elles peuvent encore réorganiser la géométrie latente, comme le montrent les deux panneaux suivants, mais elles n'améliorent presque plus la complétion masquée.

### Phase A — `latent_std`

Le second panneau mesure la dispersion moyenne des dimensions latentes :

$$
\operatorname{latent\_std}
=
\frac{1}{d}
\sum_{j=1}^{d}
\operatorname{std}(z_{:,j}).
$$

Un collapse complet ferait tendre cette quantité vers zéro : toutes les observations seraient encodées par presque le même vecteur.

Ici, `latent_std` augmente rapidement puis se stabilise autour de `1`, qui est précisément la dispersion encouragée par VICReg-lite. Cela indique que les dimensions latentes restent actives.

Il faut néanmoins éviter une conclusion trop rapide : une variance élevée ne garantit pas que l'information soit utile. Un encodeur pourrait produire des latents très dispersés mais sans rapport avec la dynamique. `latent_std` est donc une **alarme anti-collapse**, pas une métrique de contrôle.

La combinaison intéressante est :

- loss validation faible ;
- `latent_std` non nul et stable ;
- rang effectif croissant.

Ensemble, ces trois signaux montrent que le modèle ne réduit pas sa loss en produisant une représentation constante.

### Phase A — Rang effectif

Le troisième panneau mesure combien de directions latentes sont réellement utilisées. À partir des valeurs singulières $\sigma_j$ du batch centré, on construit une distribution normalisée :

$$
p_j = \frac{\sigma_j}{\sum_k \sigma_k},
\qquad
\operatorname{rank}_{\mathrm{eff}}
=
\exp\!\left(-\sum_j p_j\log p_j\right).
$$

Si toute l'information était concentrée dans une seule direction, le rang effectif serait proche de `1`. Si plusieurs dimensions portaient une information distincte, il augmenterait.

Ici, il passe approximativement de `4` à plus de `9`. Le latent de dimension 24 n'utilise pas toutes ses dimensions de manière uniforme, mais sa structure devient progressivement plus riche.

Cette croissance continue alors que la loss masquée a déjà atteint son plateau. Cela signifie que les dernières updates ne rendent plus nécessairement la prédiction plus précise, mais continuent à **redistribuer l'information** dans l'espace latent.

Ce phénomène est à lire avec prudence. Un rang plus élevé est généralement préférable à un collapse, mais le rang maximal n'est pas un objectif en soi. Pour Pendulum, l'état physique n'a que trois dimensions. Une représentation de rang 24 ne serait pas automatiquement meilleure : elle pourrait simplement dupliquer ou complexifier l'information.

---

## Phase B — Teacher forcing

Le premier panneau de la seconde ligne mesure la prédiction à un pas depuis un vrai latent :

$$
\mathcal L_{\mathrm{TF}}
=
\frac{1}{K}
\sum_{k=0}^{K-1}
\left\|
P_\phi(z_{t+k},a_{t+k})
-
z_{t+k+1}
\right\|_1.
$$

Toutes les variantes voient leur loss diminuer fortement. Le prédicteur apprend donc bien à transformer un latent courant et une action en latent suivant.

Les courbes ne doivent cependant pas être comparées comme si elles partageaient exactement la même unité. Chaque régime apprend une géométrie latente différente :

- `stage-wise` et `stage-wise-fair` partagent le même encodeur gelé ;
- `joint` continue à déplacer son espace latent pendant l'apprentissage ;
- `random-frozen` utilise une projection aléatoire fixe.

Une loss plus basse dans `random-frozen` ne signifie donc pas nécessairement que son modèle du monde est meilleur. Sa représentation peut simplement être plus facile à approximer numériquement ou avoir une échelle locale différente.

La comparaison la plus propre est celle entre `stage-wise` et `stage-wise-fair`, puisqu'ils utilisent exactement le même encodeur et commencent avec le même prédicteur. Leurs courbes se superposent pendant les 1800 premières updates. `stage-wise-fair` continue ensuite jusqu'à 3000 updates, mais le gain devient très faible : le prédicteur était déjà proche de son plateau.

### Phase B — Rollout ouvert sur le train

Le panneau central mesure la prédiction récursive. Le modèle reçoit un vrai latent au début, puis consomme ses propres sorties :

$$
\hat z_{t+k}
=
P_\phi(\hat z_{t+k-1},a_{t+k-1}).
$$

Cette loss est plus exigeante que le teacher forcing. Une petite erreur au premier pas modifie l'entrée du deuxième, puis cette erreur peut s'accumuler sur tout l'horizon.

La diminution observée montre que les prédicteurs apprennent non seulement des transitions locales, mais également des trajectoires latentes cohérentes sur plusieurs pas.

`Joint` démarre avec une loss relativement élevée, car il doit résoudre deux problèmes simultanément :

1. déplacer son espace latent vers une représentation utile ;
2. apprendre la dynamique dans cet espace encore mobile.

Le stage-wise travaille dans un espace déjà gelé. Sa cible ne change plus, ce qui rend l'optimisation du prédicteur plus stationnaire.

Malgré cette différence initiale, les courbes finissent par converger vers des valeurs proches. Le joint compense donc progressivement l'instabilité de sa représentation mobile.

### Phase B — Rollout sur validation

Le dernier panneau est le diagnostic le plus important de la seconde ligne. Il mesure le drift multi-pas sur des épisodes qui n'ont jamais servi aux gradients.

Trois situations seraient inquiétantes :

- train baisse, validation reste haute : surapprentissage des trajectoires ;
- validation commence par baisser puis remonte : entraînement trop long ;
- validation reste très supérieure au train : distribution de validation mal couverte.

Ici, les courbes de validation diminuent avec les courbes train et ne présentent pas de remontée nette. Les modèles généralisent donc leurs transitions latentes aux épisodes tenus à part.

Les courbes `stage-wise` et `stage-wise-fair` finissent presque au même niveau. Les 1200 updates supplémentaires du prédicteur apportent peu. Cela confirme que la différence finale avec `joint` ne vient probablement pas d'un prédicteur stage-wise insuffisamment entraîné.

---

## Ce que cette figure établit

La figure permet de conclure que :

1. le pré-entraînement masqué apprend une structure qui généralise ;
2. le latent ne s'effondre pas ;
3. plusieurs directions latentes restent actives ;
4. les trois types de représentation permettent d'apprendre une dynamique action-conditionnée ;
5. augmenter fortement le budget du prédicteur stage-wise apporte peu après convergence ;
6. aucune courbe ne montre de surapprentissage évident.

Mais elle ne permet pas encore de dire quel modèle contrôlera le mieux Pendulum.

Une faible loss latente peut coexister avec une mauvaise politique si :

- le latent ignore l'action ;
- les erreurs s'accumulent sur un horizon plus long que celui d'entraînement ;
- la distance au latent-but ne correspond pas au coût physique ;
- CEM exploite une région où le modèle extrapole mal.

C'est pourquoi les diagnostics suivants mesurent séparément la sensibilité à l'action, le drift selon l'horizon, la géométrie du but et, finalement, le retour dans le véritable environnement.

# Partie VI — Diagnostiquer le latent avant de planifier

Une faible loss ne garantit pas que CEM puisse choisir une action. Nous testons donc trois propriétés :
la sensibilité à l'action, le drift en boucle ouverte et l'alignement entre distance latente au but et
coût physique réel.

In [ ]:
@torch.no_grad()
def action_sensitivity(model, data, batch_size=256):
    batch = data.sample(batch_size, 1)
    z = model.encoder(batch["obs"][:, 0])
    target = model.encoder(batch["obs"][:, 1])
    actions = batch["action"][:, 0]
    permutation = torch.randperm(batch_size)
    true_error = F.l1_loss(model.predictor(z, actions), target).item()
    shuffled_error = F.l1_loss(model.predictor(z, actions[permutation]), target).item()
    return shuffled_error - true_error


@torch.no_grad()
def rollout_drift(model, data, horizon=8, n_batches=6, batch_size=128):
    errors = np.zeros(horizon)
    for _ in range(n_batches):
        batch = data.sample(batch_size, horizon)
        z = model.encoder(batch["obs"][:, 0])
        targets = model.encoder(batch["obs"][:, 1:].reshape(-1, obs_dim)).reshape(batch_size, horizon, -1)
        for step in range(horizon):
            z = model.predictor(z, batch["action"][:, step])
            errors[step] += F.l1_loss(z, targets[:, step]).item()
    return errors / n_batches


fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for name, color in [("stage-wise", "#2563eb"), ("stage-wise-fair", "#0ea5e9"), ("joint", "#d97706"), ("random-frozen", "#64748b")]:
    model = seed0[name]
    gap = action_sensitivity(model, val_buffer)
    drift = rollout_drift(model, val_buffer)
    axes[0].bar(name, gap, color=color)
    axes[1].plot(np.arange(1, len(drift) + 1), drift, marker="o", label=name, color=color)
axes[0].axhline(0, color="black", lw=1); axes[0].set_title("Information causale de l'action")
axes[0].set_ylabel("erreur shuffled - erreur vraie")
axes[1].set_title("Drift latent tenu à part"); axes[1].set_xlabel("horizon"); axes[1].set_ylabel("L1")
axes[1].legend(); fig.tight_layout()
plt.show()

## Lecture des diagnostics

Ces deux panneaux testent deux propriétés différentes du modèle action-conditionné :

- à gauche : le prédicteur utilise-t-il réellement l'action ?
- à droite : ses erreurs restent-elles maîtrisées lorsqu'il prédit plusieurs pas successifs ?

Une bonne prédiction à un pas ne suffit pas. Un modèle utilisable par CEM doit à la fois distinguer les conséquences de commandes différentes et rester cohérent lorsqu'il consomme ses propres prédictions.

### Sensibilité à l'action

Le premier panneau compare l'erreur obtenue avec les vraies actions à celle obtenue après permutation des actions dans le batch :

$$
\operatorname{gap}_{a}
=
\mathcal L\!\left(P(z_t,a_{\mathrm{mélangée}}),z_{t+1}\right)
-
\mathcal L\!\left(P(z_t,a_t),z_{t+1}\right).
$$

Si ce gap était proche de zéro, remplacer une action par celle d'une autre transition ne changerait presque rien. Le prédicteur pourrait alors ignorer l'action et extrapoler seulement depuis $z_t$. Un tel modèle pourrait avoir une faible loss, mais il serait inutilisable pour planifier : toutes les séquences proposées par CEM conduiraient presque au même futur.

Les trois gaps sont positifs : chaque prédicteur a donc appris que l'action modifie la transition. `Stage-wise` présente le gap le plus élevé. Dans son espace gelé, le prédicteur doit expliquer les différences entre transitions à travers l'action, puisqu'il ne peut pas déplacer l'encodeur pour simplifier le problème.

Le gap plus faible de `random-frozen` ne signifie pas que l'action est entièrement ignorée, mais que son effet est moins clairement organisé dans cette projection aléatoire.

### Drift en boucle ouverte

Le second panneau part d'un vrai latent $z_t$, puis applique récursivement le prédicteur :

$$
\hat z_{t+k}
=
P_\phi(\hat z_{t+k-1},a_{t+k-1}).
$$

À chaque horizon, $\hat z_{t+k}$ est comparé au latent réellement encodé depuis $o_{t+k}$. La courbe mesure donc l'accumulation des erreurs lorsque le modèle est utilisé comme il le sera par CEM.

Toutes les courbes augmentent avec l'horizon, ce qui est normal : chaque prédiction réutilise une sortie imparfaite du pas précédent. L'important est leur pente.

- `Joint` accumule ici le moins d'erreur après les premiers pas.
- `Stage-wise` reste prédictif, mais son drift augmente plus rapidement.
- `Random-frozen` finit par dériver le plus fortement.

Ce résultat nuance le panneau de gauche. `Stage-wise` rend l'effet immédiat de l'action particulièrement visible, mais `joint` construit une dynamique plus stable en rollout prolongé. Une forte sensibilité à l'action n'implique donc pas automatiquement une meilleure cohérence temporelle.

`Stage-wise-fair` permet enfin de vérifier si le drift du stage-wise vient simplement d'un prédicteur moins entraîné. S'il reste proche de `stage-wise` malgré ses updates supplémentaires, le facteur limitant est plutôt la géométrie gelée de la représentation que le budget d'optimisation du prédicteur.

## Conclusion sur l'usage de l'action et le drift

Aucune métrique ne suffit isolément :

- un gap élevé sans stabilité temporelle produit des branches très différentes mais peu fiables ;
- un drift faible avec un gap nul produit un futur stable mais insensible aux commandes ;
- un bon modèle de planning doit combiner les deux.

Ici, `stage-wise` encode plus nettement l'effet causal de l'action, tandis que `joint` offre le meilleur compromis de stabilité multi-pas. La prochaine vérification consiste à déterminer si leur géométrie latente représente correctement l’objectif physique de Pendulum.

## Géométrie du latent-but

In [ ]:
goal_obs = torch.tensor([1.0, 0.0, 0.0], dtype=torch.float32)
heldout_states = torch.tensor(val_buffer.observations())
theta = torch.atan2(heldout_states[:, 1], heldout_states[:, 0])
physical_cost = theta.square() + 0.1 * heldout_states[:, 2].square()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
goal_correlations = {}
for axis, name, color in zip(axes, ["stage-wise", "joint"], ["#2563eb", "#d97706"]):
    encoder_model = seed0[name].encoder
    with torch.no_grad():
        z = encoder_model(heldout_states)
        z_goal = encoder_model(goal_obs[None])
        latent_distance = (z - z_goal).square().sum(-1)
    correlation = float(np.corrcoef(latent_distance.numpy(), physical_cost.numpy())[0, 1])
    goal_correlations[name] = correlation
    axis.scatter(physical_cost.numpy(), latent_distance.numpy(), s=8, alpha=0.3, color=color)
    axis.set_title(f"{name} — corr={correlation:.2f}")
    axis.set_xlabel("coût physique"); axis.set_ylabel("distance au latent-but")
fig.tight_layout()
plt.show()
print("corrélations but physique / latent :", goal_correlations)

## Lecture de la géométrie du but

Sur Pendulum, CEM ne connaît pas directement le coût physique :

$$
c(o_t)=\theta_t^2+0.1\dot\theta_t^2.
$$

Il cherche uniquement à réduire la distance entre le latent imaginé et le latent de l'état-but :

$$
d_z(o_t,o_{\mathrm{goal}})
=
\left\|E(o_t)-E(o_{\mathrm{goal}})\right\|_2^2.
$$

Pour que ce planning fonctionne, cette distance doit jouer le rôle d'un **proxy du coût physique** : plus le pendule est loin de la verticale ou tourne rapidement, plus sa distance au latent-but devrait être grande.

Chaque point représente ici une observation tenue à part. L'axe horizontal donne son coût physique réel; l'axe vertical donne sa distance au latent-but. Une relation croissante signifie que CEM reçoit un paysage cohérent à optimiser.

### Géométrie `joint`

La corrélation proche de `0.90` montre un alignement fort. Les états physiquement proches du but sont généralement proches dans le latent, tandis que les états coûteux sont repoussés.

Le latent joint ne sert donc pas seulement à prédire la prochaine transition : son optimisation avec la dynamique a spontanément produit une géométrie utile au contrôle.

La relation n'est pas parfaitement linéaire, mais elle reste largement monotone. Pour CEM, c'est l'essentiel : une séquence qui réduit la distance latente a de bonnes chances d'améliorer réellement l'état du pendule.

### Géométrie `stage-wise`

La corrélation d'environ `0.68` reste positive, mais la distance sature lorsque le coût physique devient élevé. Des états très différents et très éloignés du but finissent donc par recevoir des distances latentes similaires.

Cette saturation réduit l'information fournie au planner. Si deux trajectoires mauvaises obtiennent presque le même score latent, CEM distingue difficilement laquelle commence réellement à ramener le pendule vers la verticale.

Cela ne signifie pas que la représentation stage-wise est mauvaise. Les diagnostics précédents montrent qu'elle est :

- non effondrée ;
- prédictive ;
- sensible à l'action.

Mais elle a été entraînée pour compléter des représentations masquées, pas pour organiser l'espace selon le coût de contrôle. Elle peut donc préserver les informations nécessaires à la dynamique sans faire de la distance euclidienne une bonne fonction objectif.

## Conclusion sur la géométrie du but

Cette figure met en évidence une différence fondamentale entre deux propriétés souvent confondues :

> **contenir l'information utile** n'implique pas que cette information soit organisée selon une distance utile au planning.

Le stage-wise apprend un latent informatif, mais sa géométrie gelée n'est pas explicitement façonnée pour CEM. Le joint peut au contraire adapter simultanément l'encodeur et la dynamique, ce qui semble produire ici un paysage de but plus régulier.

Cela explique pourquoi un modèle peut afficher d'excellentes losses prédictives sans obtenir automatiquement le meilleur contrôle. Le planner ne lit pas tout le contenu du latent : il n'observe que la **distance** que nous avons choisi de lui donner.

# Partie VII — CEM/MPC dans le latent

Une fois l'encodeur et le prédicteur entraînés, leurs poids restent **gelés**. Le modèle n'apprend plus pendant la planification : il devient un simulateur latent dans lequel on peut tester rapidement plusieurs futurs.

À partir de l'observation réelle $o_t$, on encode l'état courant :

$$
z_t=E(o_t).
$$

Le **Cross-Entropy Method** maintient une distribution gaussienne sur des séquences de $H$ actions. À chaque itération, il :

1. échantillonne plusieurs séquences candidates ;
2. déroule chacune dans le prédicteur action-conditionné ;
3. attribue un score aux trajectoires latentes obtenues ;
4. conserve les meilleures séquences, appelées **élites** ;
5. rapproche la moyenne et l'écart-type de la distribution de ces élites.

Pour une séquence candidate $(a_t,\ldots,a_{t+H-1})$, le futur est imaginé récursivement :

$$
\hat z_{t+h+1}=P_\phi(\hat z_{t+h},a_{t+h}),
\qquad \hat z_t=z_t.
$$

Sur Pendulum, le score récompense les trajectoires qui se rapprochent du latent-but tout en évitant des actions inutilement fortes :

$$
J =
-\lambda_{\mathrm{path}}\sum_{h=1}^{H}
\left\|\hat z_{t+h}-z_{\mathrm{goal}}\right\|_2^2
-\lambda_{\mathrm{terminal}}
\left\|\hat z_{t+H}-z_{\mathrm{goal}}\right\|_2^2
-\lambda_a\sum_{h=0}^{H-1}\|a_{t+h}\|_2^2.
$$

CEM ne calcule aucun gradient à travers le modèle. Il traite celui-ci comme un simulateur et améliore progressivement sa distribution d'actions par sélection.

Le **Model Predictive Control** ajoute ensuite une boucle de correction essentielle : bien que CEM optimise une séquence complète, seule sa première action est exécutée dans l'environnement réel. À l'instant suivant, l'agent observe le véritable nouvel état, le réencode, puis recommence toute la planification.

```text
observer → encoder → imaginer H pas → choisir → exécuter 1 pas → observer à nouveau
```

Cette replanification limite l'effet du drift. Le modèle peut se tromper sur les derniers pas de son imagination sans condamner tout l'épisode, car chaque nouvelle observation réelle replace régulièrement le planner sur la vraie trajectoire.

Le plan précédent est également décalé d'un pas pour initialiser la recherche suivante : c'est le **warm-start**. L'action imaginée pour demain devient le point de départ de la recherche actuelle, ce qui réduit le travail nécessaire à CEM.

La séparation des rôles est donc nette :

- JEPA construit la représentation ;
- le prédicteur fournit la dynamique latente ;
- CEM cherche une bonne séquence dans ce modèle ;
- MPC corrige continuellement cette imagination avec le monde réel.

In [ ]:
class LatentCEMPlanner:
    def __init__(self, horizon, action_dim, low, high, population=64, num_elites=8, iterations=3, alpha=0.5, seed=0):
        self.horizon, self.action_dim = horizon, action_dim
        self.population, self.num_elites, self.iterations = population, num_elites, iterations
        self.alpha, self.low, self.high = alpha, low.float(), high.float()
        self.init_std = ((self.high - self.low) / 2).clamp_min(1e-6)
        self.generator = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def plan(self, z0, predictor, objective, prev_mean=None):
        mean = torch.zeros(self.horizon, self.action_dim) if prev_mean is None else prev_mean.clone()
        std = self.init_std.unsqueeze(0).expand_as(mean).clone()
        for _ in range(self.iterations):
            noise = torch.randn(self.population, self.horizon, self.action_dim, generator=self.generator)
            actions = (mean[None] + std[None] * noise).clamp(
                self.low.view(1, 1, -1), self.high.view(1, 1, -1)
            )
            latent = z0.unsqueeze(0).expand(self.population, -1)
            rollout = []
            for step in range(self.horizon):
                latent = predictor(latent, actions[:, step]); rollout.append(latent)
            rollout = torch.stack(rollout, dim=1)
            elite_actions = actions[objective(rollout, actions, z0).topk(self.num_elites).indices]
            mean = (1 - self.alpha) * mean + self.alpha * elite_actions.mean(0)
            std = (1 - self.alpha) * std + self.alpha * elite_actions.std(0, correction=False).clamp_min(1e-4)
        mean = mean.clamp(self.low.view(1, -1), self.high.view(1, -1))
        return mean[0].numpy().astype(np.float32), torch.cat([mean[1:], mean[-1:]], dim=0)


def make_goal_objective(z_goal, path_weight=0.1, terminal_weight=1.0, action_weight=1e-3):
    z_goal = z_goal.detach()
    def objective(rollout_latents, actions, z0):
        distances = (rollout_latents - z_goal.view(1, 1, -1)).square().sum(-1)
        return -(path_weight * distances.sum(1) + terminal_weight * distances[:, -1]
                 + action_weight * actions.square().sum((1, 2)))
    return objective


def make_reward_objective(reward_head, continuation_head, gamma=0.99):
    def objective(rollout_latents, actions, z0):
        previous = torch.cat([z0.view(1, 1, -1).expand(len(actions), 1, -1), rollout_latents[:, :-1]], 1)
        total = torch.zeros(len(actions)); survival = torch.ones(len(actions)); discount = 1.0
        for step in range(actions.shape[1]):
            reward = reward_head(previous[:, step], actions[:, step])
            continuation = torch.sigmoid(continuation_head(previous[:, step], actions[:, step]))
            total = total + discount * survival * reward
            survival = survival * continuation
            discount *= gamma
        return total
    return objective

**Mini-test.** Une terminaison doit conserver la récompense immédiate et annuler seulement les récompenses futures.

In [ ]:
class StateReward(nn.Module):
    def forward(self, z, a): return z.squeeze(-1)

class StopImmediately(nn.Module):
    def forward(self, z, a): return torch.full((z.shape[0],), -20.0)

objective_test = make_reward_objective(StateReward(), StopImmediately(), gamma=1.0)
score_test = objective_test(
    torch.tensor([[[2.0], [5.0]]]), torch.zeros(1, 2, 1), torch.tensor([1.0])
)
assert abs(score_test.item() - 1.0) < 1e-4
planner_test = LatentCEMPlanner(5, 1, torch.tensor([-1.0]), torch.tensor([1.0]), seed=0)
print("objectif terminal OK : récompense immédiate =", round(score_test.item(), 4))

# Partie VIII — Comparaison conjointe vs stage-wise

Chaque modèle est évalué depuis les **mêmes resets**, avec un planner recréé à chaque épisode pour rester reproductible. On compare **quatre régimes** — `joint`, `stage-wise`, `stage-wise-fair` et `random-frozen` — et une politique aléatoire donne l'échelle environnementale. Les **deux comparaisons appariées** définies plus haut (compute-matched et dynamics-matched) sont reportées sous le graphique.

In [ ]:
ACTION_LOW = torch.tensor(env.action_space.low, dtype=torch.float32)
ACTION_HIGH = torch.tensor(env.action_space.high, dtype=torch.float32)
EVAL_SEEDS = [20_000 + index for index in range(8)]


@torch.no_grad()
def run_latent_controller(model, reset_seed, max_steps=200, return_trace=False):
    eval_env = gym.make(ENV_ID)
    obs, _ = eval_env.reset(seed=reset_seed)
    planner = LatentCEMPlanner(
        horizon=12, action_dim=action_dim, low=ACTION_LOW, high=ACTION_HIGH,
        population=64, num_elites=8, iterations=3, seed=reset_seed,
    )
    encoder_model, predictor_model = model.encoder, model.predictor
    z_goal = encoder_model(goal_obs[None]).squeeze(0)
    objective = make_goal_objective(z_goal)
    total, prev_mean, trace = 0.0, None, {"theta": [], "velocity": [], "action": [], "reward": []}
    for _ in range(max_steps):
        z0 = encoder_model(torch.tensor(obs, dtype=torch.float32)[None]).squeeze(0)
        action, prev_mean = planner.plan(z0, predictor_model, objective, prev_mean)
        obs, reward, terminated, truncated, _ = eval_env.step(np.clip(action, eval_env.action_space.low, eval_env.action_space.high))
        total += float(reward)
        trace["theta"].append(math.atan2(obs[1], obs[0])); trace["velocity"].append(obs[2])
        trace["action"].append(float(action[0])); trace["reward"].append(float(reward))
        if terminated or truncated: break
    eval_env.close()
    return (total, trace) if return_trace else total


def run_random_controller(reset_seed, max_steps=200):
    eval_env = gym.make(ENV_ID); obs, _ = eval_env.reset(seed=reset_seed)
    eval_env.action_space.seed(reset_seed); total = 0.0
    for _ in range(max_steps):
        obs, reward, terminated, truncated, _ = eval_env.step(eval_env.action_space.sample())
        total += float(reward)
        if terminated or truncated: break
    eval_env.close(); return total


returns = {name: [] for name in ["random", "random-frozen", "joint", "stage-wise", "stage-wise-fair"]}
for training_seed in TRAIN_SEEDS:
    for reset_seed in EVAL_SEEDS:
        for name in ["random-frozen", "joint", "stage-wise", "stage-wise-fair"]:
            returns[name].append(run_latent_controller(experiments[training_seed][name], reset_seed))
returns["random"] = [run_random_controller(seed) for seed in EVAL_SEEDS]

for name, values in returns.items():
    print(f"{name:14s}: {np.mean(values):8.1f} ± {np.std(values):6.1f}  (n={len(values)})")
    if name not in {"random"}:
        per_model = np.asarray(values).reshape(len(TRAIN_SEEDS), len(EVAL_SEEDS)).mean(1)
        print(" " * 16, "moyennes par seed modèle:", np.round(per_model, 1))
m = {k: float(np.mean(v)) for k, v in returns.items()}
print("\n— comparaisons appariées —")
print(f"compute-matched  : joint {m['joint']:.0f}  vs  stage-wise      {m['stage-wise']:.0f}")
print(f"dynamics-matched : joint {m['joint']:.0f}  vs  stage-wise-fair {m['stage-wise-fair']:.0f}")

In [ ]:
colors = {"random": "#94a3b8", "random-frozen": "#64748b", "joint": "#d97706", "stage-wise": "#2563eb", "stage-wise-fair": "#0ea5e9"}
fig, ax = plt.subplots(figsize=(8, 4.2))
names = list(returns)
for index, name in enumerate(names):
    values = np.asarray(returns[name])
    jitter = np.linspace(-0.12, 0.12, len(values))
    ax.scatter(index + jitter, values, s=24, alpha=0.65, color=colors[name])
    ax.errorbar(index, values.mean(), yerr=values.std(), fmt="o", color="black", capsize=5, lw=2)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names)
ax.set_ylabel("retour Pendulum (plus haut = meilleur)")
ax.set_title("Même expérience réelle, mêmes resets, régimes d'entraînement différents")
fig.tight_layout()
plt.show()



**Lecture.** Le résumé apparié imprimé au-dessus porte les conclusions. *Échelle* d'abord : les variantes apprises doivent battre largement `random` et surtout `random-frozen` — si l'encodeur aléatoire gelé contrôlait aussi bien, le gain ne viendrait pas de la représentation mais de la seule structure du prédicteur.

Vient ensuite la vraie question, en **deux comparaisons appariées** :

- **À compute total égal** (`joint` vs `stage-wise`) : tout apprendre ensemble laisse l'encodeur se *spécialiser* pour le contrôle, ce qui peut suffire à égaler — voire dépasser — la séparation des phases à ce budget.
- **À dynamique également entraînée** (`joint` vs `stage-wise-fair`) : c'est le test le plus juste de l'idée V-JEPA2-AC. Si `stage-wise-fair` rejoint ou dépasse `joint`, alors une représentation pré-entraînée *puis gelée* porte réellement l'information utile au contrôle ; si elle reste en retrait, c'est que le gel sacrifie une spécialisation que le joint, lui, peut exploiter.

Les nuages de points par seed comptent autant que les moyennes : une variance forte (quelques épisodes catastrophiques) indique que la conclusion dépend encore du seed et qu'il faut lire l'intervalle, pas seulement la moyenne. On rapporte donc moyenne **et** dispersion, et on évite de déclarer la tâche « résolue » sur la base d'un seul chiffre.

## Une trajectoire réelle, pas seulement une moyenne

In [ ]:
best_method = max(["joint", "stage-wise", "stage-wise-fair"], key=lambda name: np.mean(returns[name]))
best_model = experiments[0][best_method]
episode_return, trace = run_latent_controller(best_model, EVAL_SEEDS[0], return_trace=True)
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axes[0].plot(trace["theta"]); axes[0].axhline(0, color="black", ls=":"); axes[0].set_ylabel("theta")
axes[1].plot(trace["velocity"]); axes[1].set_ylabel("theta_dot")
axes[2].plot(trace["action"]); axes[2].set_ylabel("couple"); axes[2].set_xlabel("pas réel")
fig.suptitle(f"Meilleure variante : {best_method} — retour {episode_return:.1f}")
fig.tight_layout()
plt.show()

## Lecture de la trajectoire

Cette figure vérifie que l'amélioration numérique correspond à un comportement physiquement cohérent. Elle montre successivement :

- l'angle $\theta$ du pendule ;
- sa vitesse angulaire $\dot\theta$ ;
- le couple moteur choisi par CEM.

Au début, le pendule est loin de la verticale. Une action directe ne suffit pas toujours à le redresser : le contrôleur applique donc un couple important pour accumuler de l'énergie. L'angle et la vitesse augmentent pendant cette phase de **prise d'élan**.

Lorsque le pendule approche de la verticale, le couple change rapidement de signe. Le contrôleur ne cherche plus à accélérer la rotation, mais à la freiner pour éviter de dépasser l'équilibre. La vitesse angulaire présente alors un pic, puis revient progressivement vers zéro.

Après environ quelques dizaines de pas, on observe le régime recherché :

$$
\theta_t \approx 0,
\qquad
\dot\theta_t \approx 0.
$$

Le pendule reste proche de la verticale pendant le reste de l'épisode. Les actions deviennent globalement plus faibles, mais ne disparaissent pas complètement : de petites corrections restent nécessaires pour compenser les perturbations et les imperfections du modèle.

La trajectoire révèle donc trois compétences distinctes :

1. **prendre de l'élan** lorsque l'équilibre ne peut pas être atteint immédiatement ;
2. **freiner au bon moment** pour ne pas traverser la cible trop rapidement ;
3. **stabiliser** le pendule avec des corrections modérées.

C'est une preuve plus informative que le retour moyen seul. Une amélioration du score pourrait autrement provenir d'un comportement qui ralentit simplement la chute sans réussir le swing-up.

Ici, la mécanique attendue apparaît clairement : le planner exploite le modèle latent pour produire une stratégie temporelle, rejoint l'équilibre vertical, puis le maintient. Le training Pendulum a donc appris un véritable comportement de contrôle, même si les résultats agrégés montrent encore quelques échecs selon les seeds.

## Ablation — teacher forcing seul ou teacher forcing + rollout ?

In [ ]:
tf_only = train_action_model(
    experiments[0]["stage-wise"].encoder, train_buffer, val_buffer,
    action_dim, ac_cfg, seed=123, mode="tf", verbose=False,
)
tf_rollout_ablation = train_action_model(
    experiments[0]["stage-wise"].encoder, train_buffer, val_buffer,
    action_dim, ac_cfg, seed=123, mode="tf_rollout", verbose=False,
)
drift_tf = rollout_drift(tf_only, val_buffer)
drift_rollout = rollout_drift(tf_rollout_ablation, val_buffer)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(np.arange(1, len(drift_tf) + 1), drift_tf, marker="o", label="teacher forcing seul")
ax.plot(np.arange(1, len(drift_rollout) + 1), drift_rollout, marker="o", label="+ rollout ouvert")
ax.set_xlabel("horizon"); ax.set_ylabel("erreur L1 tenue à part")
ax.set_title("La loss de rollout prépare-t-elle réellement le planner ?")
ax.legend(); fig.tight_layout()
plt.show()

## Lecture de l'ablation

Cette expérience isole l'effet de la loss de rollout. Les deux modèles utilisent :

- le même encodeur stage-wise gelé ;
- la même architecture de prédicteur ;
- les mêmes données et mini-batches ;
- la même initialisation et le même nombre d'updates.

La seule différence est leur objectif :

$$
\mathcal L_{\mathrm{TF}}
\qquad\text{contre}\qquad
\mathcal L_{\mathrm{TF}}+\mathcal L_{\mathrm{rollout}}.
$$

L'axe horizontal indique la profondeur de prédiction. À l'horizon 1, les deux modèles partent d'un vrai latent. Aux horizons suivants, ils consomment progressivement leurs propres prédictions. L'axe vertical mesure l'erreur L1 par rapport aux latents réellement encodés sur la validation.

Les deux courbes augmentent avec l'horizon, ce qui est attendu : chaque erreur modifie l'entrée du pas suivant et se propage dans la trajectoire imaginée.

Le résultat intéressant est que le modèle entraîné uniquement par teacher forcing conserve ici une erreur légèrement plus faible à **tous** les horizons. Ajouter la loss de rollout n'améliore donc pas automatiquement la stabilité en boucle ouverte.

### Pourquoi le rollout peut-il dégrader le résultat ?

Au début de l'entraînement, les latents prédits sont encore mauvais. La loss de rollout demande alors au modèle de poursuivre depuis des entrées qu'il a lui-même déplacées hors de la distribution des vrais latents. Il doit simultanément :

1. apprendre la transition physique locale ;
2. corriger les erreurs produites par ses étapes précédentes.

Avec un coefficient de `1`, la loss de rollout peut recevoir autant d'importance que le teacher forcing, alors que ses cibles sont initialement beaucoup plus difficiles. Les gradients des deux objectifs peuvent alors entrer en compétition.

Le problème ressemble au *scheduled sampling* des modèles séquentiels : exposer le réseau à ses propres sorties est utile, mais le faire trop tôt ou trop fortement peut déstabiliser l'apprentissage.

Plusieurs améliorations seraient possibles :

- augmenter progressivement le poids de la loss de rollout ;
- commencer par un horizon 1, puis allonger le rollout ;
- utiliser un coefficient $\lambda_{\mathrm{rollout}}<1$ ;
- consacrer davantage d'updates au prédicteur ;
- comparer plusieurs seeds pour vérifier que l'écart est reproductible.

### Ce que l'expérience permet de conclure

Cette ablation ne démontre pas que le rollout ouvert est généralement inutile. Elle montre seulement que, **dans notre configuration**, l'ajouter avec le même poids que le teacher forcing n'améliore pas le drift.

Pendulum est également une tâche déterministe, de faible dimension, et MPC réencode l'observation réelle après chaque action. Une excellente prédiction locale peut donc suffire : les erreurs de long horizon sont régulièrement corrigées par la replanification.

Nous conservons néanmoins la loss de rollout dans la comparaison principale pour rester proches du protocole action-conditionné étudié et pour entraîner explicitement le modèle dans son régime d'utilisation par CEM. Mais nous ne lui attribuons pas artificiellement les performances obtenues.

Le résultat rappelle une règle importante : une idée théoriquement raisonnable doit être validée par une ablation. Ici, la qualité de la représentation et du teacher forcing semble compter davantage que l'ajout naïf d'une loss de rollout.

## Démonstration vidéo (Pendulum)

Pour *voir* la politique apprise, on rejoue les variantes principales avec `render_mode="rgb_array"`, puis on affiche le replay directement dans le notebook. La cellule reste séparée du training : si l'on veut seulement relire les courbes, on ne l'exécute pas. Quand on la lance, elle montre concrètement si le contrôle latent + CEM redresse puis stabilise le pendule, là où un encodeur aléatoire gelé doit échouer.

In [ ]:
PENDULUM_VIDEO_MODELS = ["joint", "stage-wise", "stage-wise-fair", "random-frozen"]

In [ ]:
def evaluate_and_display_agent(name, seed=None, max_steps=200, video_root="videos/15_action_jepa_pendulum"):
    """Évalue une variante Action-JEPA sur Pendulum et affiche un replay vidéo."""
    seed = EVAL_SEEDS[0] if seed is None else seed
    safe_label = name.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    video_env = gym.make(ENV_ID, render_mode="rgb_array")
    video_env = RecordVideo(
        video_env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == 0,
        name_prefix=f"{safe_label}_cem",
    )

    obs, _ = video_env.reset(seed=seed)
    model = experiments[0][name]
    planner = LatentCEMPlanner(
        12,
        action_dim,
        ACTION_LOW,
        ACTION_HIGH,
        population=64,
        num_elites=8,
        iterations=3,
        seed=seed,
    )
    z_goal = model.encoder(goal_obs[None]).squeeze(0)
    objective = make_goal_objective(z_goal)
    previous_mean = None
    total = 0.0
    steps = 0

    try:
        for _ in range(max_steps):
            with torch.no_grad():
                z0 = model.encoder(torch.tensor(obs, dtype=torch.float32)[None]).squeeze(0)
                action, previous_mean = planner.plan(z0, model.predictor, objective, previous_mean)
            obs, reward, terminated, truncated, _ = video_env.step(
                np.clip(action, video_env.action_space.low, video_env.action_space.high)
            )
            total += float(reward)
            steps += 1
            if terminated or truncated:
                break
    finally:
        video_env.close()

    print(f"[{name}] retour vidéo : {total:.1f} ({steps} pas)")
    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {name} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    else:
        print(f"Aucune vidéo générée pour {name}.")
    return total


pendulum_video_returns = {}
for model_name in PENDULUM_VIDEO_MODELS:
    pendulum_video_returns[model_name] = evaluate_and_display_agent(model_name)


# Partie IX — Extension HalfCheetah : changer d'objectif

Pendulum possède un objectif géométrique explicite : atteindre l'observation

$$
o_{\mathrm{goal}}=[1,0,0],
$$

qui représente un pendule vertical et immobile. Il suffit donc d'encoder cette observation et de demander à CEM de rapprocher les futurs imaginés de son latent. La récompense de l'environnement n'est pas nécessaire au planning.

HalfCheetah pose un problème différent. Le robot ne doit pas atteindre une configuration finale particulière : **courir est un comportement continu**. Une même course efficace traverse constamment des poses différentes. Choisir une observation comme latent-but pousserait le robot à reproduire une posture figée au lieu de produire un cycle de locomotion.

L'environnement est également beaucoup plus complexe :

| Propriété | HalfCheetah-v5 |
|---|---|
| Observation | 17 positions et vitesses articulaires |
| Action | 6 couples continus dans $[-1,1]$ |
| Objectif | avancer rapidement vers la droite |
| Horizon | jusqu'à 1000 pas |
| Récompense | vitesse avant moins coût de contrôle |

La récompense Gymnasium s'écrit :

$$
r_{t+1}
=
\underbrace{\frac{x_{t+1}-x_t}{\Delta t}}_{\text{vitesse vers l'avant}}
-
\underbrace{0.1\lVert a_t\rVert_2^2}_{\text{coût des couples}}.
$$

Une bonne action dépend donc de ses conséquences répétées : pousser fortement peut produire une accélération utile, mais aussi gaspiller de l'énergie ou placer le corps dans une mauvaise configuration pour les pas suivants.

## Remplacer le latent-but par un retour prédit

Sans état-but unique, CEM ne minimise plus une distance latente. Il évalue chaque séquence d'actions avec la tête de récompense :

$$
\hat r_{t+h+1}
=
R_\psi(\hat z_{t+h},a_{t+h}).
$$

La tête de continuation indique si les récompenses futures doivent encore compter :

$$
\hat c_{t+h+1}
=
\sigma\!\left(C_\omega(\hat z_{t+h},a_{t+h})\right).
$$

Le score imaginé d'une séquence devient alors un retour actualisé :

$$
J(a_{t:t+H-1})
=
\sum_{h=0}^{H-1}
\gamma^h
\left(\prod_{j=0}^{h-1}\hat c_{t+j+1}\right)
\hat r_{t+h+1}.
$$

CEM recherche les couples qui maximisent ce retour, puis MPC n'exécute que la première action avant de réencoder l'observation réelle.

Le changement est conceptuellement important :

- sur Pendulum, la représentation fournit directement la fonction objectif par sa distance au but ;
- sur HalfCheetah, la représentation fournit un état prédictif, mais une **reward head apprise** doit encore dire quels futurs sont désirables.

## Ce que teste réellement cette micro-expérience

Nous ne réalisons pas ici un entraînement complet de locomotion. Le dataset reste volontairement petit : quelques épisodes aléatoires servent à pré-entraîner l'encodeur, apprendre la dynamique et ajuster la tête de récompense.

La question testée est plus modeste :

> Une représentation pré-entraînée sans action contient-elle suffisamment d'information pour prédire une récompense de locomotion jamais utilisée pendant la Phase A ?

Pour y répondre, nous séparons les épisodes avant l'entraînement. La reward head est ajustée sur les trajectoires train, puis comparée aux récompenses d'épisodes tenus à part.

Une baisse de loss train ne suffirait pas : la tête pourrait simplement mémoriser les transitions. Nous recherchons plutôt une relation visible entre récompenses prédites et réelles sur la validation.

Cette extension vérifie donc trois branchements :

1. l'encodeur accepte un état continu de 17 dimensions ;
2. le prédicteur conditionne la dynamique sur six actions simultanées ;
3. le latent conserve assez d'information pour soutenir une tête de récompense non triviale.

Elle ne démontre pas encore que le modèle sait courir. Un véritable apprentissage HalfCheetah demanderait davantage de trajectoires, une boucle de recollecte avec le planner, une gestion de l'incertitude hors distribution et une évaluation multi-seeds face à des baselines réelles.

HalfCheetah joue ici le rôle d'un **test de généralisation structurelle** : passer d'un contrôle par latent-but à un contrôle par retour appris, sans prétendre transformer cette courte expérience en benchmark de locomotion.

In [ ]:
def halfcheetah_micro_demo():
    try:
        hc_env = gym.make("HalfCheetah-v5")
    except Exception as exc:
        print("HalfCheetah indisponible :", type(exc).__name__, exc); return None
    od, ad = hc_env.observation_space.shape[0], hc_env.action_space.shape[0]
    hc_train, hc_val = SequenceBuffer(20_000, 7), SequenceBuffer(5_000, 8)
    collect_random_episodes(hc_env, hc_train, n_episodes=4, max_steps=150, seed=30_000)
    collect_random_episodes(hc_env, hc_val, n_episodes=2, max_steps=150, seed=40_000)
    hc_rep_cfg = RepresentationConfig(latent_dim=32, hidden_dim=128, batch_size=64, mask_fraction=0.2, updates=150)
    hc_ac_cfg = ActionConfig(rollout_len=3, batch_size=64, reward_weight=1.0, ac_updates=300)
    representation = train_representation(hc_train, hc_val, od, hc_rep_cfg, seed=7)

    # train_action_model uses the global hidden width; the latent dimension is inferred from LayerNorm.
    model = train_action_model(representation["encoder"], hc_train, hc_val, ad, hc_ac_cfg, seed=7)
    with torch.no_grad():
        batch = hc_val.sample(256, 1)
        z = model.encoder(batch["obs"][:, 0])
        predicted = model.reward_head(z, batch["action"][:, 0]).numpy()
        target = batch["reward"][:, 0].numpy()
    hc_env.close()
    return model, representation["history"], predicted, target


hc_result = halfcheetah_micro_demo()
if hc_result is not None:
    hc_model, hc_rep_history, hc_predicted, hc_target = hc_result
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].plot(smooth(hc_model.history["rollout"])); axes[0].set_title("AC rollout loss")
    axes[1].scatter(hc_target, hc_predicted, s=14, alpha=0.6)
    axes[1].plot([hc_target.min(), hc_target.max()], [hc_target.min(), hc_target.max()], "k--")
    axes[1].set_xlabel("reward réelle tenue à part"); axes[1].set_ylabel("reward prédite")
    fig.tight_layout()
    plt.show()
    print("corrélation reward tenue à part :", round(float(np.corrcoef(hc_target, hc_predicted)[0, 1]), 3))
else:
    print("Micro-démo HalfCheetah sautée proprement.")

**Lecture.** La loss de rollout vérifie que la dynamique action-conditionnée apprend sur le petit dataset ; le nuage tenu à part teste une propriété plus exigeante : la tête récompense doit suivre l'ordre et l'amplitude des récompenses qu'elle n'a pas utilisées pour s'ajuster. Une corrélation positive confirme que le latent transporte un signal de locomotion exploitable, mais l'étendue limitée des actions aléatoires interdit d'en déduire qu'un long planning CEM sera déjà fiable. Cette micro-expérience valide le branchement `latent -> reward`, pas la résolution de HalfCheetah.

## Démonstration humaine — HalfCheetah

HalfCheetah n'a pas d'état-but : on planifie ici avec l'**objectif récompense** (têtes récompense + continuation) plutôt qu'un latent-but. Le même planner CEM déroule la dynamique latente et retient les couples qui maximisent le retour *imaginé*. La micro-démo n'entraîne le modèle que brièvement : on observe donc une **amorce** de comportement, pas une course experte. Drapeau séparé, car MuJoCo doit pouvoir ouvrir une fenêtre.

In [ ]:
# Replay vidéo HalfCheetah de la micro-expérience auxiliaire, si HalfCheetah est disponible.

In [ ]:
def evaluate_and_display_halfcheetah(max_steps=300, seed=0, video_root="videos/15_action_jepa_halfcheetah"):
    if hc_result is None:
        print("HalfCheetah indisponible : micro-expérience non exécutée.")
        return None

    video_dir = Path(video_root) / "reward_planner"
    video_dir.mkdir(parents=True, exist_ok=True)

    hc_env = gym.make("HalfCheetah-v5", render_mode="rgb_array")
    hc_env = RecordVideo(
        hc_env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == 0,
        name_prefix="halfcheetah_reward_planner",
    )

    hc_low = torch.tensor(hc_env.action_space.low, dtype=torch.float32)
    hc_high = torch.tensor(hc_env.action_space.high, dtype=torch.float32)
    hc_planner = LatentCEMPlanner(
        12,
        hc_env.action_space.shape[0],
        hc_low,
        hc_high,
        population=64,
        num_elites=8,
        iterations=3,
        seed=seed,
    )
    hc_objective = make_reward_objective(hc_model.reward_head, hc_model.continuation_head, gamma=0.99)
    obs, _ = hc_env.reset(seed=seed)
    previous_mean = None
    total = 0.0
    steps = 0

    try:
        for _ in range(max_steps):
            with torch.no_grad():
                z0 = hc_model.encoder(torch.tensor(obs, dtype=torch.float32)[None]).squeeze(0)
                action, previous_mean = hc_planner.plan(z0, hc_model.predictor, hc_objective, previous_mean)
            obs, reward, terminated, truncated, _ = hc_env.step(
                np.clip(action, hc_env.action_space.low, hc_env.action_space.high)
            )
            total += float(reward)
            steps += 1
            if terminated or truncated:
                break
    finally:
        hc_env.close()

    print(f"HalfCheetah — retour CEM (objectif récompense) : {total:.1f} ({steps} pas)")
    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos:
        print("Replay vidéo HalfCheetah :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=640))
    else:
        print("Aucune vidéo HalfCheetah générée.")
    return total


halfcheetah_video_return = evaluate_and_display_halfcheetah()




**Lecture.** Cette extension a un critère modeste mais précis : la reward head doit présenter une
relation avec les récompenses d'épisodes jamais utilisés pour l'optimisation. Une simple baisse de
loss train ne suffirait pas. Obtenir un vrai contrôleur demanderait ensuite davantage de données, une
incertitude explicite et une boucle de recollecte.

# Partie X — Ce que cette étude établit, et ce qu'elle n'établit pas

Ce notebook ne propose pas un nouvel algorithme officiel nommé « Action-JEPA ». Il propose une
**étude from-scratch** d'une question précise :

> Une représentation prédictive pré-entraînée sans action, puis gelée, aide-t-elle à apprendre un
> modèle action-conditionné utile au contrôle ?

On a comparé deux familles :

| Régime | Question testée |
|---|---|
| `joint` | Que se passe-t-il si l'encodeur et la dynamique s'adaptent ensemble au contrôle ? |
| `stage-wise` | Que gagne-t-on à pré-entraîner une représentation JEPA, la geler, puis apprendre seulement l'effet des actions ? |
| `stage-wise-fair` | Le résultat stage-wise vient-il de la représentation, ou simplement d'un prédicteur moins entraîné ? |
| `random-frozen` | Qu'obtient-on avec une projection gelée qui n'a rien appris ? |

Le protocole contrôle les données, les architectures, les resets et les budgets d'updates. Les figures
ne mesurent donc pas seulement une loss : elles vérifient successivement la santé du latent, la
sensibilité à l'action, le drift en boucle ouverte, la géométrie du but et le retour réel sous CEM/MPC.

C'est le point important du notebook : **une représentation n'est pas bonne parce qu'elle a une loss
faible**. Elle devient utile au contrôle seulement si elle vérifie quatre propriétés :

1. elle ne s'effondre pas ;
2. elle encode une dynamique prédictible ;
3. l'action modifie réellement les futurs imaginés ;
4. la distance ou la récompense utilisée par le planner a un sens dans le vrai environnement.

## Différences avec V-JEPA2-AC

| Élément | V-JEPA2-AC | Cette étude |
|---|---|---|
| Données de pré-entraînement | très grand corpus vidéo | quelques milliers de transitions vectorielles |
| Observation | images/vidéos | state tokens Gymnasium |
| Encodeur | grand ViT visuel | petit MLP |
| Masquage | tubes spatio-temporels | coordonnées masquées dans une fenêtre temporelle |
| Phase action-conditionnée | prédicteur causal sur latents vidéo gelés | prédicteur MLP résiduel sur latents gelés |
| Planning principal | image-but / goal latent | observation-but encodée sur Pendulum |
| Récompense | pas nécessaire pour le contrôle par image-but | têtes reward/continuation seulement pour l'extension HalfCheetah |
| Échelle | robotique réelle / grande vidéo | démonstration contrôlée et reproductible |

La comparaison ne cherche pas à masquer ces écarts. Elle isole au contraire l'idée transférable du
papier : **pré-entraîner une représentation prédictive, la geler, puis apprendre comment les actions
la déplacent**.

Dans V-JEPA2-AC, cette séparation est naturelle parce que l'encodeur visuel a appris sur une masse
énorme de vidéos et peut servir de géométrie générale du monde. Dans notre Pendulum vectoriel, le monde
est minuscule et entièrement observable : l'encodeur joint peut se spécialiser très vite. C'est
précisément pour cela que le test est intéressant.

## Ce que les résultats disent

Le pré-entraînement `stage-wise` fonctionne : le latent reste vivant, la phase masquée généralise sur
la validation, et le prédicteur action-conditionné exploite clairement les actions. La variante
`stage-wise` n'est donc pas une coquille vide : elle apprend bien une représentation utilisable.

Mais elle ne domine pas `joint` sur cette tâche. Les résultats racontent plutôt une histoire nuancée :

- `joint` construit une géométrie du but plus régulière pour CEM ;
- `joint` dérive moins à horizon long dans les diagnostics de rollout ;
- `stage-wise` rend l'effet immédiat de l'action plus visible ;
- `stage-wise-fair` n'améliore presque pas `stage-wise` malgré davantage d'updates du prédicteur ;
- les deux régimes apprennent un contrôle comparable sur Pendulum.

La conclusion n'est donc pas « stage-wise est meilleur » ni « joint est toujours meilleur ». La bonne
lecture est plus fine : sur une petite tâche vectorielle, le pré-entraînement JEPA séparé apprend une
représentation exploitable, mais il n'apporte pas d'avantage net par rapport à un entraînement conjoint.
Le bénéfice attendu de V-JEPA2-AC apparaît surtout quand la représentation perceptive est difficile,
riche, coûteuse à apprendre, et réutilisable.

## Limites

- Pendulum n'a que trois dimensions : il ne reproduit pas la difficulté perceptuelle d'une vidéo.
- Le masquage de coordonnées est un analogue pédagogique, pas un vrai masquage spatio-temporel.
- La distance au latent-but n'est utile que si la géométrie euclidienne reste alignée avec le but physique.
- CEM peut exploiter les erreurs du prédicteur, surtout loin des trajectoires exploratoires.
- Les cinq seeds donnent un signal plus sérieux qu'une seule courbe, mais pas une conclusion statistique définitive.
- HalfCheetah valide le branchement `latent -> reward`, pas encore un agent de locomotion complet.

Une suite naturelle serait une version pixel de Pendulum avec un petit CNN, puis une boucle de collecte
itérative : le planner agit, ses trajectoires enrichissent le dataset, le modèle est réentraîné, puis
le planner repart d'un latent mieux couvert. C'est là que la frontière entre JEPA, world models et RL
deviendrait vraiment expérimentale.

## Sources et lectures utiles

- **Assran et al. (2023), *Self-Supervised Learning from Images with a Joint-Embedding Predictive Architecture*.**  
  [arXiv:2301.08243](https://arxiv.org/abs/2301.08243) — I-JEPA : prédire des représentations de blocs masqués, sans reconstruire les pixels.

- **Bardes et al. (2024), *Revisiting Feature Prediction for Learning Visual Representations from Video*.**  
  [arXiv:2404.08471](https://arxiv.org/abs/2404.08471) — V-JEPA : extension vidéo, masquage spatio-temporel et prédiction dans l'espace latent.

- **Assran et al. (2025), *V-JEPA 2: Self-Supervised Video Models Enable Understanding, Prediction and Planning*.**  
  [arXiv:2506.09985](https://arxiv.org/abs/2506.09985) — inspiration directe : encodeur pré-entraîné gelé, prédicteur action-conditionné et planning vers un but visuel.

- **Grill et al. (2020), *Bootstrap Your Own Latent*.**  
  [arXiv:2006.07733](https://arxiv.org/abs/2006.07733) — BYOL : asymétrie online/target, EMA et stop-gradient sans négatifs explicites.

- **Chen & He (2020), *Exploring Simple Siamese Representation Learning*.**  
  [arXiv:2011.10566](https://arxiv.org/abs/2011.10566) — SimSiam : clarifie le rôle du stop-gradient dans les méthodes non contrastives.

- **Bardes, Ponce & LeCun (2021), *VICReg: Variance-Invariance-Covariance Regularization for Self-Supervised Learning*.**  
  [arXiv:2105.04906](https://arxiv.org/abs/2105.04906) — variance et covariance comme garde-fous explicites contre le collapse.

- **Finn & Levine (2017), *Deep Visual Foresight for Planning Robot Motion*.**  
  [arXiv:1610.00696](https://arxiv.org/abs/1610.00696) — exemple classique de planning avec modèle prédictif visuel, utile pour situer la famille « prédire puis planifier ».

- **Nagabandi et al. (2018), *Neural Network Dynamics for Model-Based Deep Reinforcement Learning with Model-Free Fine-Tuning*.**  
  [arXiv:1708.02596](https://arxiv.org/abs/1708.02596) — MPC avec modèles neuronaux, proche de l'esprit CEM/MPC utilisé ici.

- **Dans cette série.**  
  [Notebook 11 — PETS](./11_pets_halfcheetah_walkthrough.ipynb) pour les ensembles probabilistes et CEM/MPC ;  
  [Notebook 13 — Dreamer](./13_dreamer_walkthrough.ipynb) pour l'imagination latente ;  
  [Notebook 14 — MuZero](./14_muzero_discrete_control_walkthrough.ipynb) pour la planification dans un latent sans reconstruction.

## Récapitulatif final

Un JEPA apprend une géométrie prédictive sans reconstruire l'observation. Pour en faire un contrôleur,
il ne suffit pas que cette géométrie soit stable : il faut apprendre comment les actions la déplacent,
puis vérifier que le planner peut y lire un objectif exploitable.

C'est ce que le notebook rend visible. La phase masquée teste la représentation ; la phase
action-conditionnée teste la dynamique ; les diagnostics de drift testent l'usage en boucle ouverte ;
la géométrie du but teste l'objectif de CEM ; les retours réels testent enfin si tout cela survit au
contact de l'environnement.

La réponse expérimentale est volontairement nuancée. `Stage-wise` apprend une représentation saine et
contrôlable, mais ne bat pas clairement `joint` sur Pendulum. `Joint` profite de la simplicité de la
tâche pour organiser directement son latent autour du contrôle. `Stage-wise`, lui, montre mieux la
séparation conceptuelle inspirée de V-JEPA2-AC : apprendre d'abord une structure prédictive, puis
apprendre l'effet des actions dans cette structure.

La checklist finale tient donc en quatre questions :

1. le latent reste-t-il vivant ?
2. l'action change-t-elle réellement la prédiction ?
3. le modèle dérive-t-il lentement quand il se déroule lui-même ?
4. le planner améliore-t-il le retour dans le vrai environnement ?

Si l'une échoue, une belle loss ne suffit pas. Si les quatre tiennent, on a quelque chose de plus
intéressant qu'une courbe : un petit système prédictif, contrôlable, et diagnostiqué.

La version réutilisable reprend cette frontière dans `src/rl_from_scratch/action_jepa/` :
composants locaux, `ActionJepaAgent`, boucle d'entraînement et interfaces de planning restent séparés.
Le notebook expose les mécanismes ; le package fournit l'interface testée.